In [1]:
!pip install groq python-dotenv
!pip install langchain langchain-core langchain-groq langgraph kaggle

In [2]:
!pip install langchain_chroma langchain_community langchain-huggingface sentence-transformers
!pip install xgboost lightgbm -q
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from pathlib import Path

RAG_DIR = Path.cwd() / "rag_store"
import shutil as _shutil
if RAG_DIR.exists():
    _shutil.rmtree(RAG_DIR, ignore_errors=True)
RAG_DIR.mkdir(exist_ok=True)

RAG_TOP_K = 3       # number of similar solutions to retrieve
EDA_RAG_TOP_K = 6   # number of EDA knowledge docs to retrieve
EVAL_RAG_TOP_K = 4  # number of evaluator strategy docs to retrieve

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

import chromadb
client = chromadb.PersistentClient(path=str(RAG_DIR))
try:
    client.delete_collection("kaggle_solutions")
    print("Cleared old RAG collection")
except:
    pass

vector_store = Chroma(
    persist_directory=str(RAG_DIR),
    embedding_function=embeddings,
    collection_name="kaggle_solutions"
)

print(f"RAG initialized at {RAG_DIR}")

try:
    client.delete_collection("eda_knowledge")
    print("Cleared old EDA RAG collection")
except:
    pass

eda_store = Chroma(
    persist_directory=str(RAG_DIR / "eda_docs"),
    embedding_function=embeddings,
    collection_name="eda_knowledge"
)

print(f"EDA RAG initialized at {RAG_DIR / 'eda_docs'}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG initialized at c:\Users\Сергей\Downloads\rag_store
EDA RAG initialized at c:\Users\Сергей\Downloads\rag_store\eda_docs


In [3]:
from langchain_core.documents import Document


def add_competition_fact(
    competition: str,
    task_type: str,
    metric: str,
    target_column: str,
    submission_columns: list,
    train_notes: str,
    common_risks: str
):
    """Add a competition fact sheet to the EDA RAG store."""
    doc = Document(
        page_content=f"""
Competition: {competition}
Task type: {task_type}
Metric: {metric}
Target column: {target_column}
Submission columns: {submission_columns}
Train notes: {train_notes}
Common risks: {common_risks}
        """.strip(),
        metadata={
            "doc_type": "competition_fact",
            "competition": competition,
            "task_type": task_type,
            "metric": metric,
            "target_column": target_column,
        }
    )
    eda_store.add_documents([doc])
    print(f" Added competition fact: {competition}")


def add_eda_policy(category: str, title: str, content: str):
    """Add an EDA policy or guideline document."""
    doc = Document(
        page_content=f"""
Category: {category}
Title: {title}

{content}
        """.strip(),
        metadata={
            "doc_type": "eda_policy",
            "category": category,
            "title": title,
        }
    )
    eda_store.add_documents([doc])
    print(f" Added EDA policy: {title}")


def add_pitfall_doc(title: str, symptom: str, cause: str, fix: str):
    """Add a common pitfall document."""
    doc = Document(
        page_content=f"""
Pitfall: {title}
Symptom: {symptom}
Root cause: {cause}
Fix: {fix}
        """.strip(),
        metadata={
            "doc_type": "pitfall",
            "title": title,
        }
    )
    eda_store.add_documents([doc])
    print(f"️ Added pitfall: {title}")



add_competition_fact(
    competition="house-prices-advanced-regression",
    task_type="regression",
    metric="RMSE (root mean squared error of log-transformed prices)",
    target_column="SalePrice",
    submission_columns=["Id", "SalePrice"],
    train_notes="1460 rows, 79 features. Mix of ordinal, nominal, and numeric. Many NaN patterns are meaningful (e.g., no garage = NaN in GarageType).",
    common_risks="Log-transform target before training. Outliers in GrLivArea. Leakage if using test set for imputation."
)

add_competition_fact(
    competition="spaceship-titanic",
    task_type="binary_classification",
    metric="Accuracy",
    target_column="Transported",
    submission_columns=["PassengerId", "Transported"],
    train_notes="8693 rows, 13 features. Cabin column encodes deck/num/side. Spending columns are zero when in cryo-sleep.",
    common_risks="Cabin needs splitting into 3 features. CryoSleep creates deterministic zeros in spending columns — not real signal."
)

add_competition_fact(
    competition="used-cars-price-prediction",
    task_type="regression",
    metric="RMSE",
    target_column="price",
    submission_columns=["id", "price"],
    train_notes="Large dataset with car specs. High-cardinality: make, model. Numeric: year, mileage, engine_size.",
    common_risks="Price is right-skewed — consider log-transform. Year and mileage are strongly correlated."
)

add_competition_fact(
    competition="booking-rental-prediction",
    task_type="regression",
    metric="MSE (mean squared error)",
    target_column="target",
    submission_columns=["index", "prediction"],
    train_notes="~36k rows. Location features with high cardinality (200+ unique). Datetime column. Reviews columns with NaN. Target 0-365 range, ~36% zeros.",
    common_risks="Zero-inflated but not enough for two-stage (need >50%). High-cardinality location must be label-encoded, not one-hot. Datetime needs feature extraction. get_dummies needs dtype=int for sklearn compatibility."
)


add_eda_policy(
    category="regression",
    title="Regression EDA checklist",
    content="""For regression tasks:
- Check target distribution: skewness, outliers, zero-inflation percentage
- If target is right-skewed (skew > 2), note the skew but do NOT log-transform the target — it causes CV/eval metric mismatch
- If >50% zeros in target, recommend two-stage model (classifier + regressor)
- If 20-50% zeros in target, use a single model. Add zero-indicator features for INPUT columns that have many zeros (e.g. is_zero_amt_reviews = (df['amt_reviews'] == 0)). NEVER create is_zero_target from the target column — this is data leakage
- Check for ID-like columns (sequential integers, unique per row) — drop these
- Identify datetime columns by name patterns (date, dt, time, timestamp) and dtype
- Separate high-cardinality categoricals (>20 unique) from low-cardinality
- High-cardinality: recommend label encoding (integer codes for tree models)
- Low-cardinality: recommend pd.get_dummies(dtype=int)
- Check numeric column ranges for potential outliers
- Compute correlations with target — report top 5
- Flag columns with >30% missing values for special handling"""
)

add_eda_policy(
    category="binary_classification",
    title="Binary classification EDA checklist",
    content="""For binary classification EDA:
- Verify target has exactly 2 values and check class balance
- If imbalanced (minority < 20%), note for stratified CV and class_weight
- Use StratifiedKFold for cross-validation
- Identify ID-like columns (unique per row) — drop these
- Detect leakage-prone fields: columns that perfectly predict target
- Check if any feature has >0.95 correlation with target (potential leakage)
- Separate datetime handling from categorical handling
- Do NOT infer target from 'last numeric column' unless no metadata exists
- For text columns: check if they need TF-IDF or simple encoding
- Report class distribution as percentage"""
)

add_eda_policy(
    category="multiclass_classification",
    title="Multiclass classification EDA checklist",
    content="""For multiclass classification EDA:
- Count number of classes and report distribution
- If highly imbalanced, recommend class_weight='balanced'
- Use StratifiedKFold for cross-validation
- Check if classes are ordinal (can use OrdinalEncoder) or nominal
- Drop ID columns
- Report per-class sample counts"""
)

add_eda_policy(
    category="general",
    title="Schema validation policy",
    content="""Schema validation rules:
- Always verify column names match between train and test
- Check for columns in train not in test (potential target leakage)
- Check for columns in test not in train (may need special handling)
- Verify submission format matches expected columns
- Check index column type (integer vs string)
- Identify sample_submission.csv and verify format"""
)

add_eda_policy(
    category="general",
    title="Feature type detection policy",
    content="""How to classify columns:
- ID columns: name contains 'id', '_id', or is sequential integers with all unique values → DROP
- Datetime columns: dtype is datetime64, or name contains 'date', 'dt', 'time', 'timestamp' → EXTRACT features
- High-cardinality categorical: dtype is object AND nunique > 20 → LABEL ENCODE (integer codes)
- Low-cardinality categorical: dtype is object AND nunique <= 20 → ONE-HOT ENCODE (pd.get_dummies, dtype=int)
- Binary columns: nunique == 2 → keep as-is or map to 0/1
- Numeric columns: dtype is int/float → check for nulls, outliers, skewness
- Text columns: dtype is object AND average length > 50 chars → consider TF-IDF or drop"""
)


add_pitfall_doc(
    title="Zero-inflation misdiagnosis",
    symptom="Two-stage model (classifier + regressor) performs worse than single model",
    cause="Target has 20-40% zeros which is not enough for two-stage to help. The classifier introduces error that compounds with the regressor.",
    fix="Only use two-stage for >50% zeros. For 20-50%, use a single model. Add zero-indicator features for INPUT columns with many zeros (e.g. is_zero_reviews, is_zero_amount). NEVER from the target column — that is data leakage."
)

add_pitfall_doc(
    title="One-hot encoding explosion",
    symptom="get_dummies on high-cardinality column creates 200+ sparse columns, model is slow and overfits",
    cause="Using get_dummies on columns with >20 unique values creates too many features for the training data size.",
    fix="Use label encoding (integer codes) for high-cardinality categoricals. Tree models handle integer codes natively."
)

add_pitfall_doc(
    title="Target leakage from test set",
    symptom="CV score is great but leaderboard score is much worse",
    cause="Fitting imputer/encoder on combined train+test data, or using test statistics in feature engineering.",
    fix="Always fit on train only, transform test. Only combine train+test for label encoding categories (codes, not statistics)."
)

add_pitfall_doc(
    title="Datetime column left as object",
    symptom="Model ignores temporal patterns or crashes on string column",
    cause="Datetime column not converted — left as object dtype, then either dropped by select_dtypes or crashes get_dummies.",
    fix="Convert to datetime, extract year/month/day_of_week/days_since features, then drop the original datetime column."
)

add_pitfall_doc(
    title="Silent boolean column drop",
    symptom="One-hot encoded features disappear after select_dtypes(include=[np.number])",
    cause="pd.get_dummies() in newer pandas creates boolean columns by default. select_dtypes(include=[np.number]) does not include bool.",
    fix="Always use pd.get_dummies(dtype=int) to create integer columns instead of boolean."
)

add_pitfall_doc(
    title="Dropping high-cardinality features loses signal",
    symptom="MSE is high because location/name features were dropped entirely",
    cause="Analyst dropped columns with >50 unique values instead of encoding them, losing the most predictive features.",
    fix="Use label encoding for high-cardinality categoricals. Location/neighborhood features are often the strongest predictors."
)

add_pitfall_doc(
    title="Wrong target column detection",
    symptom="Model trains on wrong column or EDA reports wrong statistics",
    cause="EDA heuristic guessed target as 'last numeric column' but actual target has a specific name in competition description.",
    fix="Check competition metadata first. Only fall back to 'last numeric column' heuristic if no metadata exists. Common target names: target, Target, label, y, SalePrice, price."
)


add_pitfall_doc(
    title="CV-eval gap (overfitting)",
    symptom="CV MSE is much lower than evaluation MSE (e.g. CV=10000 but eval=14000, a 40% gap)",
    cause="Model overfits to training data. Common causes: max_depth too high, insufficient regularization, too many iterations without early stopping.",
    fix="Reduce max_depth by 1 (e.g. 6->5), increase l2_regularization (e.g. 0.1->0.5), ensure early_stopping=True, reduce max_iter. A 20%+ gap between CV and eval means the model memorizes training patterns that don't generalize."
)

add_pitfall_doc(
    title="Systematic prediction bias",
    symptom="Mean residual is consistently positive (underprediction) or negative (overprediction) across evaluations, e.g. mean_residual=+35",
    cause="Model fails to capture the overall level of the target. Often caused by missing important features, or train/test distribution shift.",
    fix="Add post-hoc bias correction: predictions = predictions + mean_residual. Or add features that correlate with the target mean (e.g. location-based averages, frequency encoding of high-cardinality columns). A mean_residual of +35 on a 0-365 target adds ~1200 to MSE."
)



def add_eval_strategy(
    signal: str,
    condition: str,
    title: str,
    strategy: str,
    priority: str = "normal"
):
    """Add an evaluator improvement strategy to the EDA RAG store."""
    doc = Document(
        page_content=f"""
Evaluation Strategy: {title}
Signal: {signal}
Condition: {condition}
Priority: {priority}

{strategy}
        """.strip(),
        metadata={
            "doc_type": "eval_strategy",
            "signal": signal,
            "condition": condition,
            "title": title,
            "priority": priority,
        }
    )
    eda_store.add_documents([doc])
    print(f"\U0001f4ca Added eval strategy: {title}")



add_eval_strategy(
    signal="plateau",
    condition="mse_change_pct < 2",
    title="Switch to gradient boosting library",
    strategy="Switch to LGBMRegressor or XGBRegressor as primary model (typically 5-15% better than sklearn on tabular data), or StackingRegressor with [LGBMRegressor(), XGBRegressor(), Ridge()] for even better results. Hyperparameter tweaks are exhausted at this point.",
    priority="high"
)

add_eval_strategy(
    signal="plateau",
    condition="mse_change_pct < 2",
    title="Two-stage zero-inflated model",
    strategy="Two-stage zero-inflated model: Step 1 \u2014 train a classifier (HistGradientBoostingClassifier) to predict zero vs non-zero target. Step 2 \u2014 train a regressor on only non-zero rows. Step 3 \u2014 combine: if classifier says zero, predict 0; else use regressor prediction. Only effective when target has >50% zeros.",
    priority="high"
)

add_eval_strategy(
    signal="plateau",
    condition="mse_change_pct < 2",
    title="Frequency encoding for high-cardinality categoricals",
    strategy="Add frequency encoding for high-cardinality categorical columns: map each category to its count in the training set, apply same mapping to test. This captures popularity signal that label encoding misses. Complements but does not replace label encoding.",
    priority="high"
)

add_eval_strategy(
    signal="plateau",
    condition="mse_change_pct < 2",
    title="Polynomial interaction features",
    strategy="Add polynomial interaction features (degree=2) between the top 3-5 most important numeric features. Use PolynomialFeatures(interaction_only=True) to avoid feature count blowup. Identify top features via model.feature_importances_ or permutation importance.",
    priority="high"
)

add_eval_strategy(
    signal="plateau",
    condition="mse_change_pct < 2",
    title="Error-bin targeted feature engineering",
    strategy="Use the error_by_bin from score_submission to identify the worst-predicted target range, then engineer features specifically for that range (e.g., binary flags for feature values common in that bin). Cross-reference with EDA summary to find which features correlate with the problematic bin.",
    priority="high"
)

add_eval_strategy(
    signal="bias",
    condition="mean_residual > 10 or mean_residual < -10",
    title="Bias correction via residual offset",
    strategy="If mean residual > 10: apply bias correction: predictions = predictions + mean_residual_value. If mean residual < -10: subtract offset: predictions = predictions - abs(mean_residual_value). A mean_residual of 35 on a 0-365 target adds ~1200 to MSE. Also consider adding frequency encoding for high-cardinality columns to capture target-correlated signal.",
)

add_eval_strategy(
    signal="cv_eval_gap",
    condition="eval_mse > cv_mse * 1.2",
    title="Reduce overfitting via regularization",
    strategy="When evaluation MSE exceeds CV MSE by more than 20%, the model is overfitting. Reduce max_depth by 1 (e.g. 6->5), increase l2_regularization to 0.5, ensure early_stopping=True, reduce max_iter. A 20%+ gap between CV and eval means the model memorizes training patterns that don't generalize.",
)

add_eval_strategy(
    signal="clipping",
    condition="worst_under > 500",
    title="Verify and adjust clipping range",
    strategy="When worst_under > 500, many predictions are far below the true values. Verify the clipping range matches the target distribution. Check if predictions are being clipped too aggressively or if the model systematically underpredicts for a subgroup. Inspect the target range from EDA and ensure clip bounds match.",
)

add_eval_strategy(
    signal="error_bin",
    condition="max_bin_error > 2 * min_bin_error",
    title="Error-bin aware targeted suggestions",
    strategy="When score_submission returns error_by_bin, compare errors across bins. The bin with highest mean absolute error needs targeted features. Cross-reference the worst bin with the EDA summary to understand what makes those rows hard to predict. Reference specific feature names and values from the EDA when suggesting improvements. Consider interaction features or bin-specific indicator variables.",
)

print(f"\n EDA RAG loaded: {eda_store._collection.count()} documents")


📋 Added competition fact: house-prices-advanced-regression
📋 Added competition fact: spaceship-titanic
📋 Added competition fact: used-cars-price-prediction
📋 Added competition fact: booking-rental-prediction
📘 Added EDA policy: Regression EDA checklist
📘 Added EDA policy: Binary classification EDA checklist
📘 Added EDA policy: Multiclass classification EDA checklist
📘 Added EDA policy: Schema validation policy
📘 Added EDA policy: Feature type detection policy
⚠️ Added pitfall: Zero-inflation misdiagnosis
⚠️ Added pitfall: One-hot encoding explosion
⚠️ Added pitfall: Target leakage from test set
⚠️ Added pitfall: Datetime column left as object
⚠️ Added pitfall: Silent boolean column drop
⚠️ Added pitfall: Dropping high-cardinality features loses signal
⚠️ Added pitfall: Wrong target column detection
⚠️ Added pitfall: CV-eval gap (overfitting)
⚠️ Added pitfall: Systematic prediction bias
📊 Added eval strategy: Switch to gradient boosting library
📊 Added eval strategy: Two-stage zero-infl

In [4]:
from langchain_core.documents import Document

def add_solution_to_rag(competition: str, task_type: str, description: str, code: str, notes: str):
    """Add a solution example to the RAG store"""
    doc = Document(
        page_content=f"{competition} {task_type} {description} {code} {notes}",
        metadata={
            "competition": competition,
            "task_type": task_type,
            "description": description,
            "code": code,
            "notes": notes
        }
    )
    vector_store.add_documents([doc])
    print(f"Added: {competition} ({task_type})")

add_solution_to_rag(
    competition="regression-mixed-features-template",
    task_type="regression",
    description="Template for regression with numeric, categorical, datetime, and high-cardinality text columns. Shows label encoding for location features and multi-seed ensemble.",
    code="""
import pandas as pd
import numpy as np
import os
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

y_train = train['target']
train = train.drop('target', axis=1)

if 'index' in test.columns:
    test_index = test['index'].values
else:
    test_index = test.index.values

id_cols = [c for c in train.columns if c == '_id' or c.endswith('_id')]
train = train.drop(id_cols, axis=1, errors='ignore')
test = test.drop(id_cols, axis=1, errors='ignore')

datetime_cols = [c for c in train.columns if 'date' in c.lower() or 'dt' in c.lower()]
for col in datetime_cols:
    train[col] = pd.to_datetime(train[col], errors='coerce')
    test[col] = pd.to_datetime(test[col], errors='coerce')
    reference_date = pd.Timestamp('2020-01-01')
    train[f'{col}_year'] = train[col].dt.year
    train[f'{col}_month'] = train[col].dt.month
    train[f'{col}_dow'] = train[col].dt.dayofweek
    train[f'{col}_days_since'] = (reference_date - train[col]).dt.days
    test[f'{col}_year'] = test[col].dt.year
    test[f'{col}_month'] = test[col].dt.month
    test[f'{col}_dow'] = test[col].dt.dayofweek
    test[f'{col}_days_since'] = (reference_date - test[col]).dt.days
    train = train.drop(col, axis=1)
    test = test.drop(col, axis=1)

def label_encode(train_df, test_df, col):
    combined = pd.concat([train_df[col], test_df[col]], axis=0).astype('category')
    codes = combined.cat.codes
    train_codes = codes[:len(train_df)].values
    test_codes = codes[len(train_df):].values
    return train_codes, test_codes

high_card_cols = [c for c in train.select_dtypes(include=['object']).columns if train[c].nunique() > 20]
for col in high_card_cols:
    train_codes, test_codes = label_encode(train, test, col)
    train[col] = train_codes
    test[col] = test_codes

numeric_cols = train.select_dtypes(include=[np.number]).columns
imputer = SimpleImputer(strategy='median')
train[numeric_cols] = imputer.fit_transform(train[numeric_cols])
test[numeric_cols] = imputer.transform(test[numeric_cols])

cat_cols = train.select_dtypes(include=['object']).columns.tolist()
train = pd.get_dummies(train, columns=cat_cols, drop_first=False, dtype=int)
test = pd.get_dummies(test, columns=cat_cols, drop_first=False, dtype=int)

train, test = train.align(test, join='left', axis=1, fill_value=0)

train = train.select_dtypes(include=[np.number])
test = test.select_dtypes(include=[np.number])

seeds = [42, 123, 456]
preds_all = np.zeros(len(test))

for seed in seeds:
    m1 = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=seed)
    m1.fit(train, y_train)
    preds_all += m1.predict(test)
    
    m2 = HistGradientBoostingRegressor(max_iter=500, max_depth=5, learning_rate=0.05, random_state=seed)
    m2.fit(train, y_train)
    preds_all += m2.predict(test)

final_preds = preds_all / (len(seeds) * 2)
final_preds = np.clip(final_preds, 0, 365)

submission = pd.DataFrame({'index': test_index, 'prediction': final_preds})
submission.to_csv(SUBMISSION_PATH, index=False)
print('Submission saved!')
""",
    notes="""CRITICAL RULES:
1) Separate target FIRST — NEVER create features from target (no is_zero_target, no target_mean).
2) Preserve test index BEFORE processing — use it in submission, NOT range(len()).
3) Label encode ALL high-cardinality cols (name, host_name, location) — do NOT drop them.
4) Impute BEFORE creating interaction features to avoid NaN propagation.
5) Apply transforms to train and test separately (never 'for df in [train,test]').
6) align(join='left', fill_value=0) — not 'inner'.
7) Multi-seed ensemble for stability.
8) Import SimpleImputer from sklearn.impute."""
)

add_solution_to_rag(
    competition="kaggle-common-pitfalls",
    task_type="debugging",
    description="Common errors in Kaggle regression: wrong imports, LabelEncoder crash, align losing target, high-cardinality freeze, target leakage, wrong submission index",
    code="""







""",
    notes="""Key rules: 1) separate target BEFORE align, 2) never reassign df inside for loop, 3) use align(join='left') not 'inner', 4) NEVER create features from target column, 5) preserve test index for submission, 6) impute BEFORE interaction features."""
)

add_solution_to_rag(
    competition="stacking-ensemble-regression",
    task_type="regression",
    description="Stacking ensemble with frequency encoding for high-cardinality columns. Uses StackingRegressor with HistGradientBoosting, ExtraTrees, and Ridge as base estimators.",
    code="""
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

y = train['target']
train = train.drop(columns=['target', '_id'], errors='ignore')
test = test.drop(columns=['_id'], errors='ignore')

for col in train.select_dtypes(include=['object']).columns:
    try:
        train[col + '_parsed'] = pd.to_datetime(train[col], errors='coerce')
        if train[col + '_parsed'].notna().sum() > len(train) * 0.5:
            train[col + '_year'] = train[col + '_parsed'].dt.year.fillna(0).astype(int)
            train[col + '_month'] = train[col + '_parsed'].dt.month.fillna(0).astype(int)
            train[col + '_dow'] = train[col + '_parsed'].dt.dayofweek.fillna(0).astype(int)
            test[col + '_parsed'] = pd.to_datetime(test[col], errors='coerce')
            test[col + '_year'] = test[col + '_parsed'].dt.year.fillna(0).astype(int)
            test[col + '_month'] = test[col + '_parsed'].dt.month.fillna(0).astype(int)
            test[col + '_dow'] = test[col + '_parsed'].dt.dayofweek.fillna(0).astype(int)
        train = train.drop(columns=[col, col + '_parsed'])
        test = test.drop(columns=[col, col + '_parsed'], errors='ignore')
    except Exception:
        train = train.drop(columns=[col + '_parsed'], errors='ignore')

for col in train.select_dtypes(include=['object']).columns:
    if train[col].nunique() > 20:
        freq = train[col].value_counts(normalize=True)
        train[col + '_freq'] = train[col].map(freq).fillna(0)
        test[col + '_freq'] = test[col].map(freq).fillna(0)
    combined = pd.concat([train[col], test[col]], axis=0).astype('category')
    codes = combined.cat.codes
    train[col + '_enc'] = codes[:len(train)].values
    test[col + '_enc'] = codes[len(train):].values
    train = train.drop(columns=[col])
    test = test.drop(columns=[col])


train, test = train.align(test, join='left', axis=1, fill_value=0)
train = train.select_dtypes(include=[np.number])
test = test.select_dtypes(include=[np.number])

imputer = SimpleImputer(strategy='median')
train = pd.DataFrame(imputer.fit_transform(train), columns=train.columns)
test = pd.DataFrame(imputer.transform(test), columns=test.columns)

estimators = [
    ('hgb', HistGradientBoostingRegressor(max_iter=500, max_depth=5, learning_rate=0.05, l2_regularization=0.1, early_stopping=True, random_state=42)),
    ('et', ExtraTreesRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)),
    ('ridge', Ridge(alpha=1.0)),
]
model = StackingRegressor(estimators=estimators, final_estimator=Ridge(alpha=1.0), cv=5, n_jobs=-1)

cv_scores = cross_val_score(model, train, y, cv=5, scoring='neg_mean_squared_error')
print(f'CV MSE: {-cv_scores.mean():.2f}')

model.fit(train, y)
predictions = model.predict(test)
predictions = np.clip(predictions, 0, y.max())

submission = pd.DataFrame({'index': range(len(predictions)), 'prediction': predictions})
submission.to_csv(SUBMISSION_PATH, index=False)
""",
    notes="""Stacking ensemble approach:
1) HistGradientBoosting + ExtraTrees + Ridge as base estimators, Ridge as meta-learner
2) Frequency encoding for high-cardinality columns (captures popularity signal that label encoding misses)
3) Both frequency AND label encoding — frequency for linear models, label for tree models
4) Stacking often beats single HistGradientBoosting by 5-15% on tabular data
5) Keep max_depth moderate (5 for HGB, 10 for ExtraTrees) to avoid overfitting
6) Use n_jobs=-1 for parallel training"""
)

add_solution_to_rag(
    competition="pitfall-overfitting-cv-eval-gap",
    task_type="debugging",
    description="Fix for when CV MSE is much lower than evaluation MSE. Model overfits to training data.",
    code="""

model = HistGradientBoostingRegressor(max_iter=1000, max_depth=7, learning_rate=0.05)

model = HistGradientBoostingRegressor(
    max_iter=500,
    max_depth=5,           # reduced from 7
    learning_rate=0.05,
    l2_regularization=0.5, # increased from default
    early_stopping=True,
    random_state=42
)
""",
    notes="""If eval MSE > CV MSE * 1.2: model overfits. Fix: reduce max_depth by 1-2, increase l2_regularization to 0.3-0.5, enable early_stopping. Do NOT increase model complexity when eval > CV."""
)

add_solution_to_rag(
    competition="pitfall-systematic-bias-correction",
    task_type="debugging",
    description="Fix for systematic prediction bias where mean residual is consistently positive or negative.",
    code="""

for col in ['name', 'host_name', 'location']:
    freq = train[col].value_counts(normalize=True)
    train[col + '_freq'] = train[col].map(freq).fillna(0)
    test[col + '_freq'] = test[col].map(freq).fillna(0)

predictions = model.predict(test)
predictions = predictions + 35  # adjust based on actual mean_residual
predictions = np.clip(predictions, 0, 365)
""",
    notes="""If mean_residual > 10: add bias correction (predictions += mean_residual). Also add frequency encoding for high-cardinality columns — it captures target-correlated signal that label encoding misses. A mean_residual of 35 adds ~1200 to MSE."""
)


add_solution_to_rag(
    competition="lightgbm-xgboost-regression",
    task_type="regression",
    description="Correct imports and usage of LGBMRegressor and XGBRegressor for tabular regression. Two-stage zero-inflated model.",
    code="""
from lightgbm import LGBMRegressor, LGBMClassifier
from xgboost import XGBRegressor, XGBClassifier

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

y = train['target']
train = train.drop(columns=['target'])


model = LGBMRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    num_leaves=31, min_child_samples=20,
    reg_alpha=0.1, reg_lambda=0.5,
    random_state=42, n_jobs=-1, verbose=-1
)
scores = cross_val_score(model, X_train, y, cv=5, scoring='neg_mean_squared_error')
print(f'CV MSE: {-scores.mean():.2f}')
model.fit(X_train, y)
predictions = model.predict(X_test)

clf = LGBMClassifier(n_estimators=300, max_depth=5, random_state=42, verbose=-1)
y_binary = (y > 0).astype(int)
clf.fit(X_train, y_binary)

non_zero_mask = y > 0
reg = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42, verbose=-1)
reg.fit(X_train[non_zero_mask], y[non_zero_mask])

zero_prob = clf.predict_proba(X_test)[:, 0]  # probability of being zero
reg_pred = reg.predict(X_test)
predictions = np.where(zero_prob > 0.5, 0, reg_pred)
predictions = np.clip(predictions, 0, y.max())
""",
    notes="""CRITICAL: LightGBM and XGBoost are separate packages, NOT part of sklearn.
CORRECT: from lightgbm import LGBMRegressor
WRONG: from sklearn.ensemble import LGBMRegressor (this will ImportError)
Two-stage zero-inflated: 1) LGBMClassifier for zero/nonzero, 2) LGBMRegressor on nonzero subset.
Always set verbose=-1 to suppress LightGBM output noise.
Use reg_lambda (L2) and reg_alpha (L1) for regularization in LightGBM."""
)

add_solution_to_rag(
    competition="safe-label-encoding-unseen-categories",
    task_type="preprocessing",
    description="Handle high-cardinality categoricals when test set contains categories not seen in training. Avoid LabelEncoder crash.",
    code="""

def safe_label_encode(train_series, test_series):
    combined = pd.concat([train_series, test_series], axis=0).astype('category')
    codes = combined.cat.codes  # unseen categories get consistent codes
    train_codes = codes[:len(train_series)].values
    test_codes = codes[len(train_series):].values
    return train_codes, test_codes

for col in high_card_cols:
    train[col], test[col] = safe_label_encode(train[col], test[col])

for col in high_card_cols:
    combined = pd.concat([train[col], test[col]])
    codes, _ = pd.factorize(combined)
    train[col] = codes[:len(train)]
    test[col] = codes[len(train):]

for col in high_card_cols:
    freq = train[col].value_counts()
    train[col + '_freq'] = train[col].map(freq).fillna(0)
    test[col + '_freq'] = test[col].map(freq).fillna(0)  # unseen -> 0
""",
    notes="""NEVER use sklearn LabelEncoder for high-cardinality columns when test may have unseen categories.
Use pd.concat + .astype('category').cat.codes or pd.factorize instead.
These handle unseen test categories safely without crashing.
Also add frequency encoding as a complementary feature (captures popularity signal)."""
)

add_solution_to_rag(
    competition="polynomial-features-no-duplicates",
    task_type="feature-engineering",
    description="Safely add polynomial interaction features without creating duplicate column names. LightGBM and XGBoost crash on duplicate feature names.",
    code="""

from sklearn.preprocessing import PolynomialFeatures

top_features = ['total_host', 'avg_reviews', 'amt_reviews']
top_features = [f for f in top_features if f in train.columns and f in test.columns]

poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)

poly_train = poly.fit_transform(train[top_features])
poly_test = poly.transform(test[top_features])
poly_names = [f'poly_{n}' for n in poly.get_feature_names_out(top_features)]

n_orig = len(top_features)
poly_train_new = poly_train[:, n_orig:]  # only interaction columns
poly_test_new = poly_test[:, n_orig:]
poly_names_new = poly_names[n_orig:]

for j, name in enumerate(poly_names_new):
    train[name] = poly_train_new[:, j]
    test[name] = poly_test_new[:, j]

train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]
""",
    notes="""LightGBM and XGBoost crash on duplicate feature names.
When using PolynomialFeatures: 1) prefix names with 'poly_' to avoid clashes,
2) skip the first N columns (they are the original features, already in your dataframe),
3) always deduplicate columns before training: df.loc[:, ~df.columns.duplicated()].
This prevents the 'Feature X appears more than once' error."""
)


Added: regression-mixed-features-template (regression)
Added: kaggle-common-pitfalls (debugging)
Added: stacking-ensemble-regression (regression)
Added: pitfall-overfitting-cv-eval-gap (debugging)
Added: pitfall-systematic-bias-correction (debugging)
Added: lightgbm-xgboost-regression (regression)
Added: safe-label-encoding-unseen-categories (preprocessing)
Added: polynomial-features-no-duplicates (feature-engineering)


In [5]:
add_solution_to_rag(
    competition="generic-feature-engineering-patterns",
    task_type="feature_engineering",
    description="Common feature engineering patterns for tabular regression: datetime extraction, ratio features, log transforms, interaction features.",
    code="""
df['date_col'] = pd.to_datetime(df['date_col'], errors='coerce')
df['year'] = df['date_col'].dt.year
df['month'] = df['date_col'].dt.month
df['day_of_week'] = df['date_col'].dt.dayofweek
df['days_since'] = (pd.Timestamp.now() - df['date_col']).dt.days
df = df.drop('date_col', axis=1)

df['ratio_feature'] = df['numerator_col'] / (df['denominator_col'].replace(0, 1))

df['log_col'] = np.log1p(df['skewed_col'])

df['interaction'] = df['col_a'] * df['col_b']

from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
print(f'CV RMSE: {-scores.mean():.2f} (+/- {scores.std():.2f})')
""",
    notes="Generic feature engineering: 1) Extract datetime features, 2) Create ratio features between related columns, 3) Log-transform skewed features, 4) Interaction features, 5) Always cross-validate."
)

add_solution_to_rag(
    competition="model-tuning-regression",
    task_type="model_selection",
    description="Model selection and tuning patterns for regression: GradientBoosting, HistGradientBoosting, cross-validation, hyperparameters",
    code="""
from sklearn.ensemble import GradientBoostingRegressor
model = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=5,
    subsample=0.8, random_state=42
)

from sklearn.ensemble import HistGradientBoostingRegressor
model = HistGradientBoostingRegressor(
    max_iter=500, learning_rate=0.05, max_depth=6,
    random_state=42
)

from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=5, scoring='neg_root_mean_squared_error')
print(f'CV RMSE: {-scores.mean():.2f}')

model.fit(X, y)
predictions = model.predict(X_test)
""",
    notes="Model tuning: prefer HistGradientBoosting for speed and built-in NaN handling. Always cross-validate. subsample=0.8 helps prevent overfitting."
)

Added: generic-feature-engineering-patterns (feature_engineering)
Added: model-tuning-regression (model_selection)


In [6]:
add_solution_to_rag(
    competition="house-prices-advanced-regression",
    task_type="regression",
    description="Predict house sale prices. Uses log transform on skewed target, feature engineering on lot/area features, encoding of neighborhood categories.",
    code="""
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

y_train = np.log1p(train['SalePrice'])
train = train.drop('SalePrice', axis=1)

test_ids = test['Id'].values  # preserve for submission
train = train.drop('Id', axis=1, errors='ignore')
test = test.drop('Id', axis=1, errors='ignore')

train['YearsSinceBuilt'] = 2024 - train['YearBuilt']
test['YearsSinceBuilt'] = 2024 - test['YearBuilt']
train['YearsSinceRemodel'] = 2024 - train['YearRemodAdd']
test['YearsSinceRemodel'] = 2024 - test['YearRemodAdd']

train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']
test['TotalSF'] = test['TotalBsmtSF'] + test['1stFlrSF'] + test['2ndFlrSF']
train['TotalBath'] = train['FullBath'] + 0.5 * train['HalfBath']
test['TotalBath'] = test['FullBath'] + 0.5 * test['HalfBath']
train['log_LotArea'] = np.log1p(train['LotArea'])
test['log_LotArea'] = np.log1p(test['LotArea'])
train['log_GrLivArea'] = np.log1p(train['GrLivArea'])
test['log_GrLivArea'] = np.log1p(test['GrLivArea'])

numeric_cols = train.select_dtypes(include=[np.number]).columns
imputer = SimpleImputer(strategy='median')
train[numeric_cols] = imputer.fit_transform(train[numeric_cols])
test[numeric_cols] = imputer.transform(test[numeric_cols])

cat_cols = train.select_dtypes(include=['object']).columns.tolist()
low_card = [c for c in cat_cols if train[c].nunique() <= 30]
high_card = [c for c in cat_cols if train[c].nunique() > 30]
train = train.drop(high_card, axis=1, errors='ignore')
test = test.drop(high_card, axis=1, errors='ignore')
train = pd.get_dummies(train, columns=low_card, dtype=int)
test = pd.get_dummies(test, columns=low_card, dtype=int)
train, test = train.align(test, join='left', axis=1, fill_value=0)

seeds = [42, 123, 456, 789, 1024]
all_preds = []
for seed in seeds:
    model = HistGradientBoostingRegressor(
        max_iter=600, learning_rate=0.05, max_depth=6,
        min_samples_leaf=20, l2_regularization=0.3, random_state=seed
    )
    model.fit(train, y_train)
    all_preds.append(model.predict(test))

predictions = np.expm1(np.mean(all_preds, axis=0))
predictions = np.clip(predictions, 0, None)
submission = pd.DataFrame({'Id': test_ids, 'SalePrice': predictions})
submission.to_csv(SUBMISSION_PATH, index=False)
""",
    notes="House prices: 1) log1p on skewed target, expm1 on predictions, 2) area interaction features (TotalSF, TotalBath), 3) apply features to train/test separately (never 'for df in [train,test]'), 4) align with join='left' not 'inner', 5) multi-seed ensemble"
)
print("Added house prices regression RAG example")

Added: house-prices-advanced-regression (regression)
Added house prices regression RAG example


In [7]:
add_solution_to_rag(
    competition="used-cars-price-prediction",
    task_type="regression",
    description="Predict used car prices. Shows handling of mixed types: mileage, engine size, year, brand encoding, geographic features.",
    code="""
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
test_index = test.index.values  # preserve for submission

y_train = train['price']
train = train.drop('price', axis=1)

drop_cols = [c for c in train.columns if 'id' in c.lower() and train[c].nunique() > len(train) * 0.5]
train = train.drop(drop_cols, axis=1, errors='ignore')
test = test.drop(drop_cols, axis=1, errors='ignore')

for col in ['registration_date', 'listing_date']:
    if col in train.columns:
        train[col] = pd.to_datetime(train[col], errors='coerce')
        test[col] = pd.to_datetime(test[col], errors='coerce')
        train[f'{col}_year'] = train[col].dt.year
        test[f'{col}_year'] = test[col].dt.year
        train[f'{col}_month'] = train[col].dt.month
        test[f'{col}_month'] = test[col].dt.month
        train = train.drop(col, axis=1)
        test = test.drop(col, axis=1)

train['car_age'] = 2024 - train.get('year', pd.Series([2020]*len(train)))
test['car_age'] = 2024 - test.get('year', pd.Series([2020]*len(test)))
train['log_mileage'] = np.log1p(train['mileage'])
test['log_mileage'] = np.log1p(test['mileage'])
train['mileage_per_year'] = train['mileage'] / (train['car_age'] + 1)
test['mileage_per_year'] = test['mileage'] / (test['car_age'] + 1)

def label_encode(train_df, test_df, col):
    combined = pd.concat([train_df[col], test_df[col]], axis=0).astype('category')
    codes = combined.cat.codes
    return codes[:len(train_df)].values, codes[len(train_df):].values

cat_cols = train.select_dtypes(include=['object']).columns.tolist()
high_card = [c for c in cat_cols if train[c].nunique() > 20]
low_card = [c for c in cat_cols if train[c].nunique() <= 20]

for col in high_card:
    train[f'{col}_enc'] = label_encode(train, test, col)
    train = train.drop(col, axis=1)
    test = test.drop(col, axis=1)

numeric_cols = train.select_dtypes(include=[np.number]).columns
imputer = SimpleImputer(strategy='median')
train[numeric_cols] = imputer.fit_transform(train[numeric_cols])
test[numeric_cols] = imputer.transform(test[numeric_cols])

train = pd.get_dummies(train, columns=low_card, dtype=int)
test = pd.get_dummies(test, columns=low_card, dtype=int)
train, test = train.align(test, join='left', axis=1, fill_value=0)

model = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, max_depth=6, random_state=42)
scores = cross_val_score(model, train, y_train, cv=3, scoring='neg_root_mean_squared_error')
print(f"CV RMSE: {(-scores.mean()):.4f}")
model.fit(train, y_train)
predictions = model.predict(test)
predictions = np.clip(predictions, 0, None)
submission = pd.DataFrame({'index': test_index, 'prediction': predictions})
submission.to_csv(SUBMISSION_PATH, index=False)
""",
    notes="Used cars: 1) auto-detect ID columns by uniqueness ratio, 2) car_age from year, 3) mileage_per_year ratio feature, 4) target encode high-cardinality cols, 5) apply features to train/test separately, 6) align with join='left'"
)

add_solution_to_rag(
    competition="zero-inflated-regression",
    task_type="regression",
    description="Predict target with many zeros. Uses two-stage model: classifier for zero/non-zero, then regressor for non-zero values. Common in booking/availability prediction.",
    code="""
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.impute import SimpleImputer

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

y_train = train['target']
train = train.drop('target', axis=1)

zero_pct = (y_train == 0).mean()
print(f"Zero-inflation: {zero_pct*100:.1f}% of targets are zero")

if zero_pct > 0.2:
    y_binary = (y_train > 0).astype(int)
    clf = GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42)
    clf.fit(train, y_binary)
    
    non_zero_mask = y_train > 0
    reg = GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)
    reg.fit(train[non_zero_mask], y_train[non_zero_mask])
    
    prob_nonzero = clf.predict_proba(test)[:, 1]
    reg_pred = reg.predict(test)
    predictions = prob_nonzero * reg_pred
else:
    model = GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)
    model.fit(train, y_train)
    predictions = model.predict(test)

predictions = np.clip(predictions, 0, 365)
""",
    notes="Two-stage model for zero-inflated targets: 1) check if >20% zeros, 2) classifier predicts P(non-zero), 3) regressor predicts value given non-zero, 4) final = P(non-zero) * E[value|non-zero]. Works well for booking/availability data."
)

print("Added used cars + zero-inflated RAG examples")

Added: used-cars-price-prediction (regression)
Added: zero-inflated-regression (regression)
Added used cars + zero-inflated RAG examples


In [8]:
add_solution_to_rag(
    competition="real-estate-target-encoding",
    task_type="feature_engineering",
    description="Smoothed target encoding for location features. Prevents overfitting on rare categories by blending category mean with global mean.",
    code="""
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

def target_encode_column(train_df, test_df, col, target, alpha=10):
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    stats['encoded'] = (stats['mean'] * stats['count'] + global_mean * alpha) / (stats['count'] + alpha)
    
    train_encoded = pd.Series(index=train_df.index, dtype=float)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for tr_idx, val_idx in kf.split(train_df):
        fold_stats = train_df.iloc[tr_idx].groupby(col)[target].agg(['mean', 'count'])
        fold_stats['encoded'] = (fold_stats['mean'] * fold_stats['count'] + global_mean * alpha) / (fold_stats['count'] + alpha)
        train_encoded.iloc[val_idx] = train_df.iloc[val_idx][col].map(fold_stats['encoded']).fillna(global_mean)
    
    test_encoded = test_df[col].map(stats['encoded']).fillna(global_mean)
    
    return train_encoded, test_encoded
""",
    notes="Target encoding with smoothing: encodes location/categorical features using target mean, regularized to prevent overfitting. Use CV on train to avoid leakage. alpha=10 controls smoothing strength."
)

print(" Added target encoding RAG example")


Added: real-estate-target-encoding (feature_engineering)
✅ Added target encoding RAG example


In [9]:
add_solution_to_rag(
    competition="real-estate-mixed-types",
    task_type="data_preprocessing",
    description="Handle mixed types: categorical, numeric, datetime for real estate prediction. Shows label encoding for high-cardinality location.",
    code="""
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
test_index = test.index.values  # preserve for submission

X = train.drop(['target'], axis=1, errors='ignore')
y = train['target']
X_test = test.copy()

date_cols = [col for col in X.columns if 'date' in col.lower() or 'dt' in col.lower()]
for col in date_cols:
    if col in X.columns:
        X[col] = pd.to_datetime(X[col], errors='coerce')
        X[f'{col}_year'] = X[col].dt.year
        X[f'{col}_month'] = X[col].dt.month
        X = X.drop(col, axis=1)
    if col in X_test.columns:
        X_test[col] = pd.to_datetime(X_test[col], errors='coerce')
        X_test[f'{col}_year'] = X_test[col].dt.year
        X_test[f'{col}_month'] = X_test[col].dt.month
        X_test = X_test.drop(col, axis=1)

def label_encode(train_df, test_df, col):
    combined = pd.concat([train_df[col], test_df[col]], axis=0).astype('category')
    codes = combined.cat.codes
    return codes[:len(train_df)].values, codes[len(train_df):].values

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
high_card = [c for c in cat_cols if X[c].nunique() > 20]
low_card = [c for c in cat_cols if X[c].nunique() <= 20]

for col in high_card:
    X[f'{col}_enc'], X_test[f'{col}_enc'] = label_encode(X, X_test, col)
    X = X.drop(col, axis=1)
    X_test = X_test.drop(col, axis=1)

numeric_cols = X.select_dtypes(include=[np.number]).columns
imputer = SimpleImputer(strategy='median')
X[numeric_cols] = imputer.fit_transform(X[numeric_cols])
X_test[numeric_cols] = imputer.transform(X_test[numeric_cols])

remaining_cat = X.select_dtypes(include=['object']).columns.tolist()
X = pd.get_dummies(X, columns=remaining_cat, dtype=int)
X_test = pd.get_dummies(X_test, columns=remaining_cat, dtype=int)

X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({'index': test_index, 'prediction': predictions})
submission.to_csv('submission.csv', index=False)
""",
    notes="Key: 1) separate target FIRST, 2) datetime features applied separately, 3) target-encode high-cardinality (>20 unique), 4) get_dummies for low-cardinality, 5) align with join='left' (not 'inner')."
)

print("Added mixed-types example to RAG")

Added: real-estate-mixed-types (data_preprocessing)
Added mixed-types example to RAG


In [ ]:
from typing import TypedDict, List, Annotated, Dict, Optional
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
import kagglehub
from pathlib import Path
import json, os, operator, tempfile, glob
os.environ['KAGGLE_USERNAME'] = "maximbochkov"
os.environ['KAGGLE_KEY'] = ""
from kaggle.api.kaggle_api_extended import KaggleApi

kaggle_api = KaggleApi()
kaggle_api.authenticate()

_NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) or os.getcwd()
os.makedirs(_NOTEBOOK_DIR, exist_ok=True)
print(f" Notebook Directory: {_NOTEBOOK_DIR}")


📁 Notebook Directory: c:\Users\Сергей\Downloads


In [ ]:
os.environ['GROQ_API_KEY'] = ''
QWEN_MODEL = "qwen/qwen3-32b"  # or "qwen-2.5-72b" if available
print(f" Using model: {QWEN_MODEL}")
COMPETITION_NAME = "mws-ai-agents-2026"
WORK_DIR = os.path.join(_NOTEBOOK_DIR, "competitions", COMPETITION_NAME)
os.makedirs(WORK_DIR, exist_ok=True)
print(f" Working Directory: {WORK_DIR}")

_cleanup_patterns = [
    os.path.join(WORK_DIR, "solution_*.py"),
    os.path.join(WORK_DIR, "submission*.csv"),
    os.path.join(WORK_DIR, "submission_best_ever.csv"),
]
_removed = 0
for _pat in _cleanup_patterns:
    for _f in glob.glob(_pat):
        os.remove(_f)
        _removed += 1
if _removed:
    print(f" Cleaned up {_removed} old file(s) from {WORK_DIR}")
else:
    print(" No old files to clean up")


📌 Using model: qwen/qwen3-32b
📁 Working Directory: c:\Users\Сергей\Downloads\competitions\mws-ai-agents-2026
🧹 Cleaned up 14 old file(s) from c:\Users\Сергей\Downloads\competitions\mws-ai-agents-2026


## Security, Guardrails & Monitoring

In [13]:
import re as _re_sec
import ast as _ast_sec
import time as _time_sec
import logging
from datetime import datetime
from collections import defaultdict
from typing import Dict, List, Optional, Any
from pathlib import Path as _Path_sec
from functools import wraps
from langchain_core.messages import AIMessage


class InputValidator:
    """Validates inputs at system boundaries.

    In this notebook all prompts are hardcoded, so prompt-injection guards
    are unnecessary.  The real risks come from:
      - a mis-typed COMPETITION_NAME (flows into Kaggle API & file paths),
      - LLM-generated output that is used as file paths or shell args.
    """

    @classmethod
    def validate_llm_output_as_path(cls, path: str, allowed_root: str) -> tuple:
        """Validate that an LLM-generated file path stays inside WORK_DIR.
        Prevents path-traversal if the model hallucinates paths like '../../../etc/passwd'."""
        try:
            resolved = _Path_sec(path).resolve()
            allowed = _Path_sec(allowed_root).resolve()
            if not str(resolved).startswith(str(allowed)):
                return False, f"Path escape blocked: {resolved} outside {allowed}"
            return True, None
        except Exception as e:
            return False, f"Invalid path: {e}"



print(" InputValidator ready")



class CodeGuardrails:
    """Static analysis of LLM-generated Python code to block dangerous operations
    before execution. Uses AST inspection + regex fallback for string evasion."""

    BLOCKED_IMPORTS = {
        'socket', 'http.server', 'xmlrpc', 'ftplib', 'smtplib',
        'telnetlib', 'ctypes', 'multiprocessing',
        'webbrowser', 'antigravity', 'turtle', 'signal',
    }

    BLOCKED_CALLS = {
        'eval', 'exec', 'compile', '__import__',
        'os.system', 'os.popen', 'os.execv', 'os.execve',
        'os.spawn', 'os.spawnl', 'os.spawnle',
        'os.kill', 'os.killpg',
        'shutil.rmtree',
        'subprocess.call', 'subprocess.run', 'subprocess.Popen',
        'subprocess.check_output', 'subprocess.check_call',
    }

    BLOCKED_ATTRS = {
        '__subclasses__', '__bases__', '__globals__',
        '__builtins__', '__code__', '__reduce__',
    }

    MAX_CODE_SIZE = 50_000  # characters
    MAX_FILE_WRITES = 5

    @classmethod
    def check_code(cls, code: str, work_dir: str = None) -> tuple:
        """
        Analyze Python code for security issues.
        Returns (is_safe: bool, issues: List[str], severity: str).
        severity: 'critical' (block), 'warning' (log only), 'ok'
        """
        issues = []
        severity = 'ok'

        if len(code) > cls.MAX_CODE_SIZE:
            return False, [f"Code too large: {len(code)} chars (max {cls.MAX_CODE_SIZE})"], 'critical'

        try:
            tree = _ast_sec.parse(code)
        except SyntaxError as e:
            return False, [f"SyntaxError: {e}"], 'critical'

        file_write_count = 0

        for node in _ast_sec.walk(tree):
            if isinstance(node, (_ast_sec.Import, _ast_sec.ImportFrom)):
                modules = []
                if isinstance(node, _ast_sec.Import):
                    modules = [a.name.split('.')[0] for a in node.names]
                elif node.module:
                    modules = [node.module.split('.')[0]]
                for mod in modules:
                    if mod in cls.BLOCKED_IMPORTS:
                        issues.append(f"BLOCKED import: '{mod}' (line {node.lineno})")
                        severity = 'critical'

            if isinstance(node, _ast_sec.Call):
                call_name = cls._get_call_name(node)
                if call_name:
                    if call_name in cls.BLOCKED_CALLS:
                        issues.append(f"BLOCKED call: '{call_name}' (line {node.lineno})")
                        severity = 'critical'

                    if call_name == 'open' and len(node.args) >= 2:
                        if isinstance(node.args[1], _ast_sec.Constant) and 'w' in str(node.args[1].value):
                            file_write_count += 1

                    if call_name in ('requests.get', 'requests.post', 'urllib.request.urlopen'):
                        issues.append(f"WARNING: network call '{call_name}' (line {node.lineno})")
                        if severity == 'ok':
                            severity = 'warning'

            if isinstance(node, _ast_sec.Attribute):
                if node.attr in cls.BLOCKED_ATTRS:
                    issues.append(f"BLOCKED attribute: '{node.attr}' (line {node.lineno})")
                    severity = 'critical'

        if file_write_count > cls.MAX_FILE_WRITES:
            issues.append(f"Too many file writes: {file_write_count} (max {cls.MAX_FILE_WRITES})")
            severity = 'critical'

        evasion_patterns = [
            (r'__import__\s*\(', 'dynamic __import__'),
            (r'getattr\s*\(.+["\']__', 'getattr dunder access'),
            (r'os\.system\s*\(', 'os.system call'),
        ]
        for pattern, label in evasion_patterns:
            for match in _re_sec.finditer(pattern, code):
                line_no = code[:match.start()].count('\n') + 1
                msg = f"BLOCKED string evasion: {label} (line {line_no})"
                if not any(label in i for i in issues):
                    issues.append(msg)
                    severity = 'critical'

        is_safe = severity != 'critical'
        return is_safe, issues, severity

    @classmethod
    def _get_call_name(cls, node) -> Optional[str]:
        if isinstance(node.func, _ast_sec.Name):
            return node.func.id
        elif isinstance(node.func, _ast_sec.Attribute):
            parts = []
            current = node.func
            while isinstance(current, _ast_sec.Attribute):
                parts.append(current.attr)
                current = current.value
            if isinstance(current, _ast_sec.Name):
                parts.append(current.id)
            return '.'.join(reversed(parts))
        return None

print(" CodeGuardrails ready")



class AgentMonitor:
    """Centralized monitoring: structured logging, latency tracking, error
    counting, and automatic alerts for anomalies."""

    def __init__(self):
        self.logs: List[Dict] = []
        self.metrics: Dict[str, list] = defaultdict(list)
        self.error_counts: Dict[str, int] = defaultdict(int)
        self.start_time = datetime.now()
        self._alerts: List[Dict] = []

        self.MAX_ITERATIONS = 50
        self.MAX_ERRORS_PER_AGENT = 6
        self.MAX_NODE_TIME = 400       # seconds per node call
        self.MAX_TOTAL_TIME = 3600     # 1 hour total

    def log_event(self, agent: str, event_type: str, details: str = "",
                  severity: str = "info"):
        entry = {
            "timestamp": datetime.now().isoformat(),
            "agent": agent,
            "event": event_type,
            "details": details[:1000],
            "severity": severity,
        }
        self.logs.append(entry)

        icon = {"info": "ℹ️", "warning": "️", "error": "", "critical": ""}.get(severity, "")
        print(f"  [MONITOR] {icon} [{agent}] {event_type}: {details[:200]}")

        if severity in ("error", "critical"):
            self.error_counts[agent] += 1
            if self.error_counts[agent] >= self.MAX_ERRORS_PER_AGENT:
                self._add_alert("error_threshold",
                                f"{agent} reached {self.error_counts[agent]} errors", "critical")

    def track_latency(self, agent: str, duration_sec: float):
        self.metrics[f"{agent}_latency"].append(duration_sec)
        if duration_sec > self.MAX_NODE_TIME:
            self._add_alert("slow_execution",
                            f"{agent} took {duration_sec:.1f}s (limit {self.MAX_NODE_TIME}s)", "warning")

    def track_iteration(self, iteration: int):
        if iteration > self.MAX_ITERATIONS:
            self._add_alert("max_iterations",
                            f"Iteration {iteration} exceeds limit {self.MAX_ITERATIONS}", "critical")

    def track_guardrail_violation(self, agent: str, issues: List[str]):
        for issue in issues:
            self.log_event(agent, "guardrail_violation", issue, "critical")
        self._add_alert("guardrail_violation",
                        f"{agent}: {len(issues)} violation(s) — {issues[0]}", "critical")

    def _add_alert(self, alert_type: str, message: str, severity: str):
        self._alerts.append({
            "timestamp": datetime.now().isoformat(),
            "type": alert_type,
            "message": message,
            "severity": severity,
        })
        print(f"  [ALERT]  {severity.upper()}: {message}")

    def should_stop(self) -> tuple:
        critical = [a for a in self._alerts if a['severity'] == 'critical']
        if len(critical) >= 3:
            return True, f"Halted: {len(critical)} critical alerts"
        elapsed = (datetime.now() - self.start_time).total_seconds()
        if elapsed > self.MAX_TOTAL_TIME:
            return True, f"Halted: total runtime exceeded {self.MAX_TOTAL_TIME}s"
        return False, ""

    def get_summary(self) -> str:
        elapsed = (datetime.now() - self.start_time).total_seconds()
        lines = [
            "=" * 60, "MONITORING SUMMARY", "=" * 60,
            f"Total runtime: {elapsed:.1f}s",
            f"Events logged: {len(self.logs)}",
            f"Alerts: {len(self._alerts)}",
            "", "--- Errors per agent ---",
        ]
        for agent, count in sorted(self.error_counts.items()):
            lines.append(f"  {agent}: {count}")
        lines += ["", "--- Avg latency per agent ---"]
        for key, values in sorted(self.metrics.items()):
            if '_latency' in key:
                avg = sum(values) / len(values)
                lines.append(f"  {key.replace('_latency','')}: {avg:.1f}s ({len(values)} calls)")
        if self._alerts:
            lines += ["", "--- Alerts ---"]
            for a in self._alerts:
                lines.append(f"  [{a['severity']}] {a['type']}: {a['message']}")
        return "\n".join(lines)


monitor = AgentMonitor()
print(" AgentMonitor ready")



def monitored_node(agent_name: str):
    """Decorator: wraps any agent node function with monitoring,
    latency tracking, error handling, and automatic stop checks."""
    def decorator(func):
        @wraps(func)
        def wrapper(state, *args, **kwargs):
            iteration = state.get("iteration_count", 0)
            monitor.track_iteration(iteration)

            should_stop, reason = monitor.should_stop()
            if should_stop:
                monitor.log_event(agent_name, "forced_stop", reason, "critical")
                return {
                    "messages": [AIMessage(content=f"SYSTEM HALT: {reason}")],
                    "next": "END",
                    "task_completed": True,
                }

            monitor.log_event(agent_name, "start", f"iteration={iteration}")
            t0 = _time_sec.time()
            try:
                result = func(state, *args, **kwargs)
                dt = _time_sec.time() - t0
                monitor.track_latency(agent_name, dt)
                monitor.log_event(agent_name, "complete",
                                  f"duration={dt:.1f}s next={result.get('next','?')}")
                return result
            except Exception as e:
                dt = _time_sec.time() - t0
                monitor.track_latency(agent_name, dt)
                monitor.log_event(agent_name, "exception", str(e)[:500], "error")
                return {
                    "messages": [AIMessage(content=f"{agent_name} error: {str(e)[:500]}")],
                    "next": "supervisor",
                    "iteration_count": iteration + 1,
                }
        return wrapper
    return decorator

print(" monitored_node decorator ready")
print()
print("All security, guardrails, and monitoring systems initialized.")

✅ InputValidator ready
✅ CodeGuardrails ready
✅ AgentMonitor ready
✅ monitored_node decorator ready

All security, guardrails, and monitoring systems initialized.


In [14]:
from typing import TypedDict, List, Annotated, Dict, Optional
import operator

class AgentState(TypedDict):
    messages: Annotated[List, operator.add]
    next: str
    competition_name: str
    dataset_paths: Dict[str, str]
    submission_path: Optional[str]
    task_completed: bool
    iteration_count: int
    retrieved_examples: List[Dict]
    solution_code: Optional[str]
    solution_path: Optional[str]
    code_validated: bool
    validation_errors: Optional[str]
    submission_uploaded: bool
    eda_summary: Optional[str]
    eda_rag_context: Optional[str]
    cv_score: Optional[str]
    eval_score: Optional[float]
    eval_passed: bool
    eval_trace: Optional[str]                  # ← Buffered observability trace from evaluator
    eval_improvements: Optional[List[str]]
    eval_error_by_bin: Optional[Dict]          # Per-bin error breakdown from evaluator
    coder_reasoning: Optional[str]
    error_history: Annotated[List, operator.add]
    feature_eng_code: Optional[str]
    candidate_paths: Optional[List[str]]
    best_eval_score: Optional[float]           # Best MSE seen so far
    best_solution_code: Optional[str]          # Code that produced the best score
    best_submission_path: Optional[str]        # Submission file for the best score

print("AgentState ready (with best-ever tracking)")

AgentState ready (with best-ever tracking)


## Kaggle Agent

In [ ]:
@tool
def download_competition_files(
    competition_name: str,
    download_dir: str = None
) -> dict:
    """
    Download files from a Kaggle competition using kagglehub.
    
    Args:
        competition_name: Kaggle competition slug (e.g., "digit-recognizer", "titanic")
        download_dir: Target directory (default: WORK_DIR/competitions/{competition_name})
    
    Returns:
        dict with keys: 'success' (bool), 'files' (list of ABSOLUTE paths), 'count' (int), 'state_updates' (dict)
    """
    try:
        print(f" Downloading '{competition_name}'...")
        
        cached_path = kagglehub.competition_download(competition_name)
        cached_path = Path(cached_path).resolve()
        
        base_dir = Path(download_dir).resolve() if download_dir else Path(WORK_DIR).resolve()
        if str(cached_path) != str(base_dir):
            import shutil
            base_dir.mkdir(parents=True, exist_ok=True)
            shutil.copytree(cached_path, base_dir, dirs_exist_ok=True)
        else:
            base_dir = cached_path
        
        DATA_EXTS = {".csv", ".json", ".parquet", ".zip", ".txt", ".tsv", ".xlsx"}
        absolute_paths = []
        for f in base_dir.rglob("*"):
            if f.is_file() and f.suffix.lower() in DATA_EXTS:
                absolute_paths.append(str(f.resolve()))
        
        path_dict = {Path(p).stem.lower(): p for p in absolute_paths}
        
        print(f" Downloaded {len(absolute_paths)} files to {base_dir}")
        return {
            "success": True,
            "files": absolute_paths,
            "count": len(absolute_paths),
            "state_updates": {"dataset_paths": path_dict}
        }
    except Exception as e:
        print(f" Download failed: {e}")
        return {"success": False, "error": str(e), "files": [], "state_updates": {}}


@tool
def upload_submission(
    competition_name: str,
    submission_file_path: str,
    message: str = "Auto-submission from multi-agent workflow"
) -> dict:
    """
    Upload a submission file to Kaggle leaderboard.
    
    Args:
        competition_name: Kaggle competition slug
        submission_file_path: ABSOLUTE path to CSV submission file
        message: Submission description for leaderboard
    
    Returns:
        dict with keys: 'success' (bool), 'state_updates' (dict)
    """
    try:
        file_path = Path(submission_file_path).resolve()
        if not file_path.exists():
            return {"success": False, "error": f"File not found: {file_path}", "state_updates": {}}
        
        std_path = file_path.parent / "submission.csv"
        if file_path.name != "submission.csv":
            import shutil
            shutil.copy2(file_path, std_path)
            print(f" Renamed {file_path.name} -> submission.csv")
            file_path = std_path
        
        print(f" Submitting {file_path} to '{competition_name}'")
        
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        api.competition_submit(str(file_path), message, competition_name)
        return {
            "success": True,
            "state_updates": {"submission_uploaded": True}
        }
    except Exception as e:
        print(f" Upload failed: {e}")
        return {"success": False, "error": str(e), "state_updates": {}}


In [16]:
kaggle_agent_system_prompt = """
You are a Kaggle Competition Specialist Agent.


1. download_competition_files(competition_name, download_dir) - Download competition files
2. upload_submission(competition_name, submission_file_path, message) - Upload submission CSV to Kaggle


1. Download files first before any analysis
2. If asked to upload, call upload_submission with the exact file path provided
3. Call the appropriate tool — state updates happen automatically from tool results
"""

kaggle_agent_prompt = ChatPromptTemplate.from_messages([
    ("system", kaggle_agent_system_prompt),
    MessagesPlaceholder(variable_name="messages", optional=True),
    ("human", "{input}"),
])

llm = ChatGroq(model=QWEN_MODEL, temperature=0, max_tokens=4096)

kaggle_tools = [download_competition_files, upload_submission]
llm_with_tools = llm.bind_tools(kaggle_tools)

kaggle_agent_chain = kaggle_agent_prompt | llm_with_tools


In [17]:
def kaggle_node(state: AgentState) -> dict:
    messages = state["messages"]
    competition = state.get("competition_name", "unknown")
    
    user_input = messages[-1].content
    submission_path = state.get("submission_path", "")
    if submission_path:
        user_input += f"\nSubmission file available at: {submission_path}"
    
    response = kaggle_agent_chain.invoke({"messages": messages, "input": user_input})
    
    tool_registry = {t.name: t for t in kaggle_tools}
    tool_messages = []
    collected_updates = {}
    
    for tool_call in response.tool_calls:
        tool_id = tool_call.get("id")
        tool_fn = tool_registry.get(tool_call.get("name"))
        
        print(f"\U0001f527 Executing tool: {tool_call.get('name')}")
        result = tool_fn.invoke(tool_call.get("args", {})) if tool_fn else {"error": "unknown tool"}
        
        tool_messages.append(ToolMessage(content=json.dumps(result), tool_call_id=tool_id))
        
        for key, val in result.get("state_updates", {}).items():
            if isinstance(val, dict) and isinstance(collected_updates.get(key), dict):
                collected_updates[key].update(val)
            else:
                collected_updates[key] = val
    
    dataset_paths = state.get("dataset_paths", {}).copy()
    dataset_paths.update(collected_updates.pop("dataset_paths", {}))
    
    print(f"\U0001f5c2\ufe0f  Paths in state: {list(dataset_paths.keys())}")
    
    return {
        "messages": [response] + tool_messages,
        "next": "supervisor",
        "dataset_paths": dataset_paths,
        **collected_updates,
        "iteration_count": state.get("iteration_count", 0) + 1
    }

print("\u2705 kaggle_node ready (tools declare state_updates, node merges — no if-else)")


✅ kaggle_node ready (tools declare state_updates, node merges — no if-else)


## Coder Agent

In [18]:
from langchain_core.tools import tool
import json

@tool
def retrieve_similar_solutions(query: str) -> dict:
    """
    Retrieve similar solution examples from RAG store.
    
    Args:
        query: Search query for similar solutions (e.g., "tabular regression prediction")
    
    Returns:
        dict with keys: 'success', 'examples', 'count'
    """
    try:
        docs = vector_store.similarity_search(query, k=RAG_TOP_K)
        examples = []
        for doc in docs:
            examples.append({
                "competition": doc.metadata.get("competition", ""),
                "task_type": doc.metadata.get("task_type", ""),
                "description": doc.metadata.get("description", ""),
                "code": doc.metadata.get("code", ""),
                "notes": doc.metadata.get("notes", "")
            })
        return {"success": True, "examples": examples, "count": len(examples)}
    except Exception as e:
        return {"success": False, "error": str(e), "examples": []}

In [19]:
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq
from langchain_core.callbacks import BaseCallbackHandler
from pathlib import Path
import re, json as _json

class CoderTracer(BaseCallbackHandler):
    def __init__(self):
        self.trace_lines = []
    def reset(self):
        self.trace_lines = []
    def on_llm_start(self, serialized, prompts=None, messages=None, **kwargs):
        model = serialized.get("kwargs", {}).get("model_name", serialized.get("id", ["?"])[-1])
        self.trace_lines.append(f"  [CODER LLM] Calling {model}")
        if messages:
            for batch in messages:
                for msg in batch:
                    role = msg.type if hasattr(msg, "type") else "unknown"
                    content = msg.content if hasattr(msg, "content") else str(msg)
                    preview = content[-500:].replace(chr(10), " ")
                    self.trace_lines.append(f"    >> {role} (last 500 chars): {preview}")
    def on_llm_end(self, response, **kwargs):
        for gen in response.generations:
            for g in gen:
                msg = g.message if hasattr(g, "message") else None
                if not msg:
                    continue
                content = msg.content if msg.content else ""
                import re as _re
                think = _re.search(r"<think>(.*?)</think>", content, _re.DOTALL)
                if think:
                    thinking = think.group(1).strip()
                    self.trace_lines.append(f"  [CODER THINKING] ({len(thinking)} chars): {thinking}")
                code_blocks = _re.findall(r"```python\n(.*?)\n```", content, _re.DOTALL)
                if code_blocks:
                    self.trace_lines.append(f"  [CODER OUTPUT] {len(code_blocks)} code block(s), largest={max(len(b) for b in code_blocks)} chars")
                elif content:
                    self.trace_lines.append(f"  [CODER OUTPUT] {len(content)} chars (no code blocks found)")
    def get_trace(self):
        return chr(10).join(self.trace_lines)

coder_tracer = CoderTracer()
coder_tracer_config = {"callbacks": [coder_tracer]}

coder_llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.6,
    max_tokens=32768
)

coder_agent_system_prompt = """
You are a Kaggle Coder Agent. You generate complete, executable Python solutions for machine learning competitions.

- Output a COMPLETE, SELF-CONTAINED Python script inside a single ```python``` code block.
- The script must: load data, preprocess, engineer features, train model(s), predict, save submission.csv.
- Use the EXACT file paths provided (TRAIN_PATH, TEST_PATH, SUBMISSION_PATH).
- Wrap the entire pipeline in try/except and print full traceback on error.
- You MUST compute cross-validation before training the final model. Use this exact code:
  scores = cross_val_score(model, X_train, y, cv=5, scoring='neg_mean_squared_error')
  print(f'CV MSE: {-scores.mean():.2f}')
  For StackingRegressor (which is slow with cross_val_score), use a train/val split instead:
  X_tr, X_val, y_tr, y_val = train_test_split(X_train, y, test_size=0.2, random_state=42)
  model.fit(X_tr, y_tr); val_mse = mean_squared_error(y_val, model.predict(X_val))
  print(f'CV MSE: {val_mse:.2f}')
  NEVER hardcode or fake a CV score. The score MUST come from actual computation.

- Separate target BEFORE any train/test alignment.
- Apply transforms to train and test SEPARATELY. NEVER use "for df in [train, test]: df = ..." — reassigning df does NOT modify the originals.
- After all preprocessing, add: train = train.select_dtypes(include=[np.number]); test = test.select_dtypes(include=[np.number]) as a safety net.
- Use sklearn, pandas, numpy, xgboost, and lightgbm. Prefer LGBMRegressor or XGBRegressor over HistGradientBoostingRegressor — they are typically 5-15% more accurate on tabular data.
- Do NOT create features derived from the target column — this is data leakage that causes massive regression.
  BANNED: is_zero_target, is_zero, target_mean, target_enc, (y == 0), (train['target'] == 0), or ANY expression using y/target to create a feature.
  These features exist in train but NOT in test, so the model overfits to them and predictions collapse.
- NEVER apply log1p/log transform to the TARGET variable. It changes the optimization objective and makes CV MSE meaningless compared to eval MSE. Only log-transform FEATURES, never the target.
- For high-cardinality categoricals: use LABEL ENCODING (integer codes). Trees split on these natively.
- For low-cardinality categoricals (<=5 unique): pd.get_dummies(dtype=int) is fine.
- After encoding, align train and test with join="left" and fill_value=0 (NOT join="inner" which drops columns).
- Handle datetime columns: extract year, month, day_of_week, then DROP the original column.
- Keep max_iter <= 500 and max_depth <= 7 to avoid timeout. HistGradientBoostingRegressor is fast for single models.
- When improving an existing solution that plateaus (score barely changes across iterations), switch to StackingRegressor with [HistGradientBoostingRegressor, ExtraTreesRegressor, Ridge] as estimators. Stacking often beats single models by 5-15%.


- Columns must be 'index' and 'prediction'.
- The submission creation code MUST be exactly:
  submission = pd.DataFrame({'index': range(len(predictions)), 'prediction': predictions})
- Do NOT use test_id, test_ids, test_idx, test['_id'], or any other variable for the index column.
  The index is always range(len(predictions)) — sequential integers starting from 0.
- Clip predictions to a reasonable range.

- If a previous solution is provided with its score, you MUST keep its preprocessing pipeline intact.
- Only make TARGETED changes: add 1-2 features, tune hyperparameters (learning_rate, max_depth, n_estimators), or adjust the ensemble.
- If previous score is within 20% of the threshold, make SMALL changes only — do NOT rewrite from scratch.
- NEVER remove working preprocessing steps (label encoding, datetime extraction, imputation) that the previous solution had.
- If you rewrite and get a worse score, you are wasting iterations.

- If previous errors are provided, fix those specific issues without changing unrelated code.
"""

def _build_rag_query(prev_code, error_history, validation_errors, eval_improvements, eda_summary):
    """Build a RAG search query: error-focused on retry, EDA-focused on first attempt."""
    if prev_code and (error_history or validation_errors or eval_improvements):
        query_parts = ["regression"]
        err_text = (validation_errors + " " + " ".join(error_history[-2:])).lower()
        _kw = {
            "train test feature alignment column mismatch": ["feature names", "mismatch"],
            "column not found keyerror pandas preprocessing": ["keyerror", "not in index"],
            "missing values imputation pipeline": ["nan", "imput"],
            "label encoding categorical features sklearn": ["dtype", "object", "categorical"],
            "efficient model training large dataset": ["timeout", "memory"],
        }
        for append_str, triggers in _kw.items():
            if any(t in err_text for t in triggers):
                query_parts.append(append_str)
        if eval_improvements:
            query_parts.append(" ".join(eval_improvements[:2]))
        query_parts.append("stacking ensemble frequency encoding alternative model")
        if len(query_parts) == 2:
            query_parts.append("fix runtime error sklearn pipeline")
    else:
        eda_lower = eda_summary.lower()
        query_parts = ["regression tabular prediction"]
        _kw = {
            "datetime feature extraction": ["datetime", "date", "timestamp"],
            "geographic location features": ["lat", "lon", "location", "geographic", "coordinate"],
            "high cardinality encoding target encoding": ["high-cardinality", "high cardinality", ">50 unique", ">100 unique"],
            "skewed target log transform": ["skew", "log transform"],
            "zero-inflated two-stage model": ["zero", "zero-inflat", "35%", "40%", "50%"],
        }
        for append_str, triggers in _kw.items():
            if any(t in eda_lower for t in triggers):
                query_parts.append(append_str)
    return " ".join(query_parts)


def _is_valid_python(code):
    try:
        compile(code, '<string>', 'exec')
        return True
    except SyntaxError:
        return False

def _find_best_code_block(text):
    """Find the longest valid Python code block in text."""
    all_blocks = re.findall(r'```python\n(.*?)\n```', text, re.DOTALL)
    valid = [b for b in all_blocks if _is_valid_python(b) and ('import' in b or 'pd.read_csv' in b)]
    return max(valid, key=len) if valid else None


def _build_retry_hint(error_history, validation_errors, eval_improvements, prev_score):
    """Build a dynamic retry hint from actual errors and evaluator feedback."""
    parts = []
    if error_history:
        recent = error_history[-2:]
        parts.append("ERRORS FROM PREVIOUS ATTEMPTS (fix these):")
        for i, err in enumerate(recent, 1):
            parts.append(f"  {i}) {err[:300]}")
    if validation_errors:
        parts.append(f"VALIDATION ISSUES: {validation_errors[:300]}")
    if eval_improvements:
        parts.append("EVALUATOR SUGGESTIONS (apply these):")
        for imp in eval_improvements[:4]:
            parts.append(f"  - {imp}")
    if prev_score is not None:
        parts.append(f"Previous MSE={prev_score:.0f}. Make SMALL targeted changes to improve, do NOT rewrite from scratch.")
    if not parts:
        parts.append("Improve the previous solution with better hyperparameters and feature engineering.")
    return " ".join(parts)


def _extract_code_and_reasoning(response, candidates):
    """Extract thinking/reasoning and valid Python code from LLM response."""
    content = response.content
    think_match = re.search(r'<think>(.*?)</think>', content, flags=re.DOTALL)
    if think_match:
        reasoning = think_match.group(1).strip()
        clean = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL).strip()
    else:
        unclosed = re.search(r'<think>(.*)', content, flags=re.DOTALL)
        if unclosed:
            raw_thinking = unclosed.group(1).strip()
            before_think = content[:unclosed.start()].strip()
            after_blocks = re.findall(r'```python\n(.*?)\n```', raw_thinking, re.DOTALL)
            if after_blocks and not before_think:
                code_cands = [b for b in after_blocks if 'import' in b or 'pd.read_csv' in b]
                best = max(code_cands, key=len) if code_cands else after_blocks[-1]
                clean = '```python' + chr(10) + best + chr(10) + '```'
            else:
                clean = before_think if before_think else raw_thinking
            reasoning = raw_thinking
            print("\u26a0\ufe0f Unclosed <think> tag detected")
        else:
            reasoning = ""
            clean = content.strip()

    if reasoning:
        print(f" Coder reasoning ({len(reasoning)} chars):")
        print(f"    {reasoning}")



    code_match = re.search(r'```python\n(.*?)\n```', clean, re.DOTALL)
    code = code_match.group(1) if code_match else clean

    if not _is_valid_python(code) or ('import' not in code and 'pd.read_csv' not in code):
        print("\u26a0\ufe0f Extracted code is NOT valid Python — searching candidates")
        best = None
        for c in candidates:
            found = _find_best_code_block(c.content)
            if found and (best is None or len(found) > len(best)):
                best = found
        if best:
            code = best
            print(f"\u2705 Found valid code block ({len(code)} chars)")
        else:
            for c in candidates:
                raw = re.sub(r'<think>.*?</think>', '', c.content, flags=re.DOTALL).strip()
                raw = re.sub(r'<think>.*', '', raw, flags=re.DOTALL).strip()
                if _is_valid_python(raw) and ('import' in raw or 'pd.read_csv' in raw):
                    code = raw
                    print(f"\u2705 Used raw response as code ({len(code)} chars)")
                    break
            else:
                print("\u274c No valid Python code found in any candidate")

    if len(code) > 8000:
        all_blocks = re.findall(r'```python\n(.*?)\n```', clean, re.DOTALL)
        if all_blocks:
            real = [b for b in all_blocks if 'import' in b or 'pd.read_csv' in b]
            code = max(real, key=len) if real else all_blocks[-1]

    return code, reasoning


def _postprocess_code(code):
    """Fix common LLM-generated code bugs."""
    
    code = re.sub(
        r'\{-([^}]+?):([\.\d]*[fd])\}',
        lambda m: '{(' + '-' + m.group(1) + '):' + m.group(2) + '}',
        code
    )
    
    return code


def _save_candidates(extracted_code, candidates, competition, iteration=0):
    """Save primary + extra candidate solutions to disk. Returns (solution_path, all_paths)."""
    solution_filename = f"solution_{competition.replace('-', '_')}.py"
    solution_path = Path(WORK_DIR) / solution_filename
    
    with open(solution_path, "w", encoding="utf-8") as f:
        f.write(extracted_code)
    
    iter_path = Path(WORK_DIR) / f"solution_{competition.replace('-', '_')}_iter{iteration}.py"
    with open(iter_path, "w", encoding="utf-8") as f:
        f.write(extracted_code)
    print(f" Snapshot: {iter_path.name}")
    
    all_candidate_paths = [str(solution_path.resolve())]
    for c_idx, c_resp in enumerate(candidates[1:], 2):
        try:
            c_content = c_resp.content
            c_clean = re.sub(r'<think>.*?</think>', '', c_content, flags=re.DOTALL).strip()
            if not c_clean:
                c_clean = re.sub(r'<think>', '', c_content, flags=re.DOTALL).strip()
            c_match = re.search(r'```python\n(.*?)\n```', c_clean, re.DOTALL)
            if c_match:
                c_code = _postprocess_code(c_match.group(1))
                c_path = Path(WORK_DIR) / f"solution_candidate_{c_idx}.py"
                with open(c_path, "w", encoding="utf-8") as f:
                    f.write(c_code)
                all_candidate_paths.append(str(c_path.resolve()))
                print(f" Candidate {c_idx} saved: {c_path}")
        except Exception as e:
            print(f"️ Candidate {c_idx} extraction failed: {e}")
    
    print(f" Generated {len(all_candidate_paths)} candidate solutions")
    print(f" Primary: {solution_path}")
    return solution_path, all_candidate_paths

def coder_node(state: AgentState) -> dict:
    messages = state["messages"]
    competition = state.get("competition_name", "unknown")
    dataset_paths = state.get("dataset_paths", {})
    
    train_path = dataset_paths.get("train", "")
    test_path = dataset_paths.get("test", "")
    
    print(f"\n Coder Agent: {competition}")
    print(f"   Train: {train_path}")
    print(f"   Test: {test_path}")
    
    eda_summary = state.get("eda_summary", "")
    error_history = state.get("error_history", [])
    validation_errors = state.get("validation_errors", "") or ""
    eval_improvements = state.get("eval_improvements", []) or []
    prev_code = state.get("solution_code", "")
    
    NL = chr(10)
    query = _build_rag_query(prev_code, error_history, validation_errors, eval_improvements, eda_summary)
    print(f" Searching RAG: {query}")
    
    docs = vector_store.similarity_search(query, k=RAG_TOP_K)
    
    retrieved_examples = []
    rag_context = ""
    for doc in docs:
        retrieved_examples.append({
            "competition": doc.metadata.get("competition", ""),
            "task_type": doc.metadata.get("task_type", ""),
            "code": doc.metadata.get("code", ""),
            "notes": doc.metadata.get("notes", "")
        })
        rag_context += f"""
Notes: {doc.metadata.get('notes', '')}
```python
{doc.metadata.get('code', '')}
```
"""
        print(f" RAG: {doc.metadata.get('competition')}")
    if not docs:
        print(" No similar solutions in RAG")
    
    if eval_improvements:
        pitfall_query = "pitfall " + " ".join(eval_improvements[:2])
        pitfall_docs = vector_store.similarity_search(pitfall_query, k=2)
        seen = {doc.metadata.get("competition") for doc in docs}
        for doc in pitfall_docs:
            comp = doc.metadata.get("competition", "")
            if comp not in seen:
                seen.add(comp)
                retrieved_examples.append({
                    "competition": comp,
                    "task_type": doc.metadata.get("task_type", ""),
                    "code": doc.metadata.get("code", ""),
                    "notes": doc.metadata.get("notes", "")
                })
                rag_context += NL + f"## RAG PITFALL: {comp} ({doc.metadata.get('task_type', '')})" + NL
                rag_context += f"Notes: {doc.metadata.get('notes', '')}" + NL
                rag_context += "```python" + NL + doc.metadata.get('code', '') + NL + "```" + NL
                print(f" RAG pitfall: {comp}")
    
    best_code = state.get("best_solution_code", "")
    best_score = state.get("best_eval_score")
    prev_code = best_code if best_code else state.get("solution_code", "")
    prev_score = best_score if best_score is not None else state.get("eval_score")
    is_retry = bool(prev_code) and _is_valid_python(prev_code)
    
    file_paths_block = "## FILE PATHS (copy-paste these EXACTLY, do NOT invent paths):" + NL + f'TRAIN_PATH = r"{train_path}"' + NL + f'TEST_PATH = r"{test_path}"' + NL + f'SUBMISSION_PATH = r"{str(Path(WORK_DIR) / "submission.csv")}"'
    
    if is_retry:
        user_message = file_paths_block
        
        if eval_improvements:
            user_message += NL + NL + "## FIX THESE SPECIFIC ISSUES FIRST (from evaluator):" + NL
            for imp in eval_improvements:
                user_message += "- " + str(imp) + NL
            user_message += NL + "Apply ONLY the changes listed above. Do NOT rewrite the solution."
        
        if prev_score is not None:
            score_info = f"scored MSE={prev_score:.0f}"
        else:
            score_info = "score unknown"
        header = NL + NL + f"## PREVIOUS SOLUTION ({score_info}, threshold={MSE_THRESHOLD}). Make ONLY targeted changes:"
        code_block = "```python" + NL + prev_code[:6000] + NL + "```"
        user_message += header + NL + code_block
        
        user_message += NL + NL + "## TASK INSTRUCTION:" + NL + messages[-1].content
        
        if error_history:
            user_message += NL + NL + "## PREVIOUS ERRORS (do NOT repeat):" + NL
            for i, err in enumerate(error_history[-2:], 1):
                user_message += f"{i}) {err[:200]}" + NL
    else:
        user_message = file_paths_block
        user_message += NL + NL + "## TASK INSTRUCTION:" + NL + messages[-1].content
        
        if error_history:
            user_message += NL + NL + "## PREVIOUS ERRORS (do NOT repeat these mistakes):" + NL
            for i, err in enumerate(error_history[-3:], 1):
                user_message += f"Attempt {i}: {err}" + NL
        
        if eda_summary:
            user_message += NL + NL + "## EDA INSIGHTS (from data analysis — use these to guide your approach):" + NL + eda_summary
        
        feature_eng_code = state.get("feature_eng_code", "")
        if feature_eng_code:
            user_message += NL + NL + "## FEATURE ENGINEERING (integrate into your solution):" + NL + "```python" + NL + feature_eng_code + NL + "```"
    
    is_retry = is_retry
    
    supervisor_instruction = messages[-1].content if messages else ""
    print(f"    [CODER] Supervisor instruction ({len(supervisor_instruction)} chars):")
    print(f"    {supervisor_instruction[:500]}")
    if "EXECUTION ERRORS" in supervisor_instruction:
        print("    [CODER] Has execution errors: YES")
    if "EVAL FEEDBACK" in supervisor_instruction:
        print("    [CODER] Has eval feedback: YES")
    if "EDA ANALYSIS" in supervisor_instruction:
        print("    [CODER] Has EDA analysis: YES")
    if is_retry:
        print(f"    [CODER] is_retry=True, prev_code={len(prev_code)} chars, prev_score={prev_score}")
        if "PREVIOUS SOLUTION" not in user_message:
            print("    [CODER] WARNING: PREVIOUS SOLUTION block MISSING!")
    if error_history:
        print(f"    [CODER] error_history: {len(error_history)} entries")
    print(f"    [CODER] Total user_message: {len(user_message)} chars")
    
    if is_retry:
        fresh_user_message = file_paths_block
        fresh_user_message += NL + NL + "## TASK INSTRUCTION:" + NL + messages[-1].content
        if eval_improvements:
            fresh_user_message += NL + NL + "## EVALUATOR FEEDBACK (previous attempt had these issues, avoid them):" + NL
            for imp in eval_improvements:
                fresh_user_message += "- " + str(imp) + NL
        if eda_summary:
            fresh_user_message += NL + NL + "## EDA INSIGHTS:" + NL + eda_summary
        if rag_context:
            fresh_user_message += NL + NL + rag_context
    else:
        fresh_user_message = user_message
        if rag_context:
            fresh_user_message += NL + NL + rag_context
    
    if is_retry:
        strategies = [
            {"temp": 0.5, "hint": _build_retry_hint(error_history, validation_errors, eval_improvements, prev_score)},
            {"temp": 0.8, "hint": (
                "Try a COMPLETELY DIFFERENT model architecture: "
                "1) Use sklearn.ensemble.StackingRegressor with [HistGradientBoostingRegressor, ExtraTreesRegressor, Ridge] as estimators. "
                "2) Or try BaggingRegressor with n_estimators=20 wrapping HistGradientBoostingRegressor. "
                "3) Add frequency encoding for high-cardinality columns (count occurrences in train). "
                "4) Do NOT create any features derived from the target column. "
                "5) Use test dataframe original index for submission."
            )},
        ]
    else:
        strategies = [
            {"temp": 0.3, "hint": (
                "Focus on robust preprocessing and a clean model. "
                "IMPORTANT RULES: 1) Do NOT create features derived from the target column (no is_zero, no target_mean, etc). "
                "2) Use the test dataframe index for submission index column, NOT range(). "
                "3) Impute missing values BEFORE creating interaction features. "
                "4) Use label encoding for high-cardinality categoricals."
            )},
            {"temp": 0.5, "hint": (
                "Use HistGradientBoostingRegressor with careful tuning. "
                "IMPORTANT: 1) No target-derived features as model inputs. "
                "2) If using log1p on target, remember np.expm1 on predictions AND compute CV on raw scale. "
                "3) Preserve test index for submission. "
                "4) Consider frequency encoding for name/host_name/location columns."
            )},
        ]
    candidates = []
    for idx, strat in enumerate(strategies):
        try:
            strat_llm = ChatGroq(model="qwen/qwen3-32b", temperature=strat["temp"], max_tokens=32768)
            base_msg = fresh_user_message if idx == 1 else user_message
            strat_message = base_msg + chr(10) + chr(10) + "## STRATEGY HINT:" + chr(10) + strat["hint"]
            resp = strat_llm.invoke([
                {"role": "system", "content": coder_agent_system_prompt},
                {"role": "user", "content": strat_message}
            ], config=coder_tracer_config)
            candidates.append(resp)
            print(f"Candidate {idx+1}/{len(strategies)} generated (temp={strat['temp']}, retry={is_retry})")
        except Exception as e:
            print(f"Candidate {idx+1} failed: {e}")
    if not candidates:
        candidates = [coder_llm.invoke([
            {"role": "system", "content": coder_agent_system_prompt},
            {"role": "user", "content": user_message}
        ], config=coder_tracer_config)]
    
    response = candidates[0]  # default to first, will pick best after extraction
    
    extracted_code, coder_reasoning = _extract_code_and_reasoning(response, candidates)

    extracted_code = _postprocess_code(extracted_code)

    if not extracted_code or not _is_valid_python(extracted_code):
        print(" Coder failed to generate valid Python code")
        coder_trace = coder_tracer.get_trace()
        if coder_trace:
            print(chr(10) + "--- Coder LLM Trace ---")
            print(coder_trace)
            print("--- End Trace ---")
        coder_tracer.reset()
        return {
            "messages": [AIMessage(content="Coder failed to generate valid Python code — retry needed")],
            "next": "supervisor",
            "retrieved_examples": retrieved_examples,
            "solution_code": "",
            "coder_reasoning": coder_reasoning,
            "solution_path": state.get("solution_path", ""),
            "candidate_paths": [],
            "code_validated": False,
            "validation_errors": "Code extraction failed — LLM produced no valid Python",
            "iteration_count": state.get("iteration_count", 0) + 1,
        }

    solution_path, all_candidate_paths = _save_candidates(extracted_code, candidates, competition, state.get("iteration_count", 0))
    
    
    coder_trace = coder_tracer.get_trace()
    if coder_trace:
        print(chr(10) + "--- Coder LLM Trace ---")
        print(coder_trace)
        print("--- End Trace ---")
    coder_tracer.reset()
    clean_output = re.sub(r"<think>.*?</think>", "", response.content, flags=re.DOTALL).strip()
    clean_output = re.sub(r"<think>.*", "", clean_output, flags=re.DOTALL).strip()
    if not clean_output:
        clean_output = f"Code generated for {competition}"
    return {
        "messages": [AIMessage(content=clean_output)],
        "next": "supervisor",
        "retrieved_examples": retrieved_examples,
        "solution_code": extracted_code,
        "coder_reasoning": coder_reasoning,
        "solution_path": str(solution_path.resolve()),
        "candidate_paths": all_candidate_paths,
        "code_validated": False,
        "validation_errors": None,
        "iteration_count": state.get("iteration_count", 0) + 1,
    }

print(" Coder Agent ready (autonomous — discovers approach from EDA + RAG)")

✅ Coder Agent ready (autonomous — discovers approach from EDA + RAG)


## EDA Agent

In [20]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

def get_column_info(train_path: str) -> str:
    """Extract column names, dtypes, and statistics from train.csv - shows ALL columns"""
    try:
        train_path = Path(train_path).resolve()
        
        if not train_path.exists():
            return "️ Could not read train.csv (file not found)"
        
        df = pd.read_csv(train_path, nrows=100)
        
        info = []
        for i, col in enumerate(df.columns, 1):
            dtype = str(df[col].dtype)
            sample = str(df[col].dropna().iloc[0])[:50] if len(df[col].dropna()) > 0 else "N/A"
            null_count = int(df[col].isna().sum())
            unique_count = int(df[col].nunique())
            
            if 'date' in col.lower() or 'dt' in col.lower() or 'time' in col.lower():
                col_type = " DATETIME"
            elif dtype in ['int64', 'float64', 'int32', 'float32']:
                col_type = " NUMERIC"
            elif dtype == 'object':
                col_type = " CATEGORICAL"
            else:
                col_type = dtype
            
            info.append(f"  {i:2d}. {col:20s}: {col_type:15s} ({dtype:10s}), nulls: {null_count:3d}, unique: {unique_count:4d}, sample: {sample}")
        
        return "\n".join(info)
    
    except Exception as e:
        return f"️ Could not read schema: {str(e)}"

print(" get_column_info() updated with full statistics")

✅ get_column_info() updated with full statistics


In [21]:
import re
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq
import pandas as pd
import numpy as np

eda_llm = ChatGroq(model="qwen/qwen3-32b", temperature=0, max_tokens=4096)

eda_system_prompt = """
You are a Data Analysis Agent. You analyze raw data and produce a STRUCTURED actionable summary.

You will receive:
1. Raw statistics computed from the data
2. Retrieved EDA knowledge (competition facts, policies, pitfall warnings) from a knowledge base

Use the retrieved knowledge to guide your analysis — it contains best practices, common pitfalls, and competition-specific notes. Apply relevant policies and flag any pitfalls that match the data.

Your output MUST include these sections:

State if this is regression or classification, and the target column name.

List column count, train/test shape differences, and any schema mismatches.

If >50% of target values are zero: "TARGET IS HEAVILY ZERO-INFLATED ({X}% zeros) — recommend two-stage model."
If 20-50% zeros: "TARGET HAS MODERATE ZEROS ({X}%) — use single model. Add zero-indicator features for INPUT columns with many zeros (e.g. is_zero_amt_reviews). NEVER create is_zero_target from the target — this is leakage."
Otherwise: "Target is NOT zero-inflated."

List ID columns and name columns to drop. Do NOT drop high-cardinality categoricals — use label encoding instead.

List these columns with their cardinality. Recommend: "Use label encoding (integer codes) for these columns (do NOT drop, do NOT use get_dummies)."

List columns suitable for pd.get_dummies(dtype=int) one-hot encoding.

List datetime columns and what features to extract (year, month, day_of_week, days_since_reference).

List numeric columns with their ranges and null counts.

Describe the target: mean, median, min, max, % zeros, skewness.

List top 5 features correlated with target.

Suggest specific interaction features, ratios, log transforms based on the data.

List any pitfalls from the retrieved knowledge that apply to this dataset. For each, state the pitfall name and the specific fix.

Recommend model type and approach based on data characteristics.
Use sklearn, xgboost (XGBRegressor), or lightgbm (LGBMRegressor). Prefer LGBMRegressor or XGBRegressor for best accuracy.

Be concise and specific. Use exact column names from the data.
"""


def retrieve_eda_context(competition: str, task_type: str, raw_stats: str) -> str:
    """Retrieve relevant EDA knowledge from the EDA RAG store, then filter through LLM."""
    
    query_llm = ChatGroq(model="qwen/qwen3-32b", temperature=0, max_tokens=256)
    query_response = query_llm.invoke([
        {"role": "system", "content": "You generate search queries for a knowledge base. Given data statistics, output a SHORT search query (under 30 words) that captures the key data characteristics: task type, feature types, target distribution issues, encoding needs. Output ONLY the query, nothing else."},
        {"role": "user", "content": f"Competition: {competition}\nTask type: {task_type}\nData statistics:\n{raw_stats[:1500]}"}
    ])
    query = re.sub(r"<think>.*?</think>", "", query_response.content, flags=re.DOTALL).strip()
    print(f"   EDA RAG query (LLM-generated): {query}")
    
    try:
        docs = eda_store.similarity_search(query, k=EDA_RAG_TOP_K)
    except Exception as e:
        print(f"  ️ EDA RAG retrieval failed: {e}")
        return ""
    
    if not docs:
        print("   No EDA knowledge found in RAG")
        return ""
    
    facts = []
    policies = []
    pitfalls = []
    
    for doc in docs:
        doc_type = doc.metadata.get("doc_type", "unknown")
        if doc_type == "competition_fact":
            facts.append(doc.page_content)
        elif doc_type == "eda_policy":
            policies.append(doc.page_content)
        elif doc_type == "pitfall":
            pitfalls.append(doc.page_content)
        else:
            policies.append(doc.page_content)
    
    retrieved_text = ""
    if facts:
        retrieved_text += "### SIMILAR COMPETITION FACTS:\n" + "\n---\n".join(facts) + "\n\n"
    if policies:
        retrieved_text += "### EDA POLICIES:\n" + "\n---\n".join(policies) + "\n\n"
    if pitfalls:
        retrieved_text += "### KNOWN PITFALLS:\n" + "\n---\n".join(pitfalls) + "\n\n"
    
    print(f"   Retrieved {len(docs)} docs: {len(facts)} facts, {len(policies)} policies, {len(pitfalls)} pitfalls")
    
    filter_llm = ChatGroq(model="qwen/qwen3-32b", temperature=0, max_tokens=2048)
    filter_response = filter_llm.invoke([
        {"role": "system", "content": "You are a relevance filter. Given data statistics and retrieved knowledge documents, select ONLY the documents that are relevant to THIS specific dataset. Remove irrelevant ones. Output the relevant documents as-is, grouped by type. Be concise."},
        {"role": "user", "content": f"Data statistics:\n{raw_stats[:2000]}\n\nRetrieved knowledge:\n{retrieved_text}\n\nReturn ONLY the relevant documents for this dataset. Keep the original text."}
    ])
    
    filtered = re.sub(r"<think>.*?</think>", "", filter_response.content, flags=re.DOTALL).strip()
    print(f"   LLM filtered EDA context ({len(filtered)} chars)")
    
    return filtered


def eda_node(state: AgentState) -> dict:
    dataset_paths = state.get("dataset_paths", {})
    train_path = dataset_paths.get("train", "")
    test_path = dataset_paths.get("test", "")
    competition = state.get("competition_name", "unknown")
    
    print(f"\n EDA Agent: analyzing {train_path}")
    
    try:
        df = pd.read_csv(train_path)
        
        stats_lines = []
        
        target_col = None
        for candidate in ['target', 'Target', 'label', 'y', 'CLASS']:
            if candidate in df.columns:
                target_col = candidate
                break
        if target_col is None:
            num_cols = df.select_dtypes(include=[np.number]).columns
            target_col = num_cols[-1] if len(num_cols) > 0 else df.columns[-1]
        
        stats_lines.append(f"TARGET COLUMN: {target_col}")
        t = df[target_col]
        stats_lines.append(f"TARGET: mean={t.mean():.1f}, std={t.std():.1f}, min={t.min()}, max={t.max()}, median={t.median():.1f}")
        
        zero_pct = (t == 0).mean() * 100
        stats_lines.append(f"TARGET DISTRIBUTION: zeros={int((t==0).sum())} ({zero_pct:.1f}%), max_val({int(t.max())})={int((t==t.max()).sum())} ({(t==t.max()).mean()*100:.1f}%)")
        if zero_pct > 50:
            stats_lines.append(f"*** ZERO-INFLATED TARGET: {zero_pct:.1f}% of values are zero — consider two-stage model ***")
        
        numeric = df.select_dtypes(include=[np.number])
        corr = numeric.corr()[target_col].drop(target_col).sort_values(key=abs, ascending=False)
        stats_lines.append(f"CORRELATIONS WITH TARGET: {', '.join([f'{c}={v:.3f}' for c, v in corr.items()])}")
        
        try:
            test_df = pd.read_csv(test_path, nrows=5)
            train_only = set(df.columns) - set(test_df.columns)
            test_only = set(test_df.columns) - set(df.columns)
            if train_only:
                stats_lines.append(f"COLUMNS IN TRAIN ONLY: {sorted(train_only)}")
            if test_only:
                stats_lines.append(f"COLUMNS IN TEST ONLY: {sorted(test_only)}")
            stats_lines.append(f"SHAPES: train={df.shape}, test={test_df.shape}")
        except Exception:
            stats_lines.append(f"SHAPE: {df.shape[0]} rows x {df.shape[1]} columns")
        
        for col in df.columns:
            n = df[col].nunique()
            dtype = str(df[col].dtype)
            nulls = int(df[col].isnull().sum())
            card_note = ""
            if dtype == 'object' and n > 50:
                card_note = " *** HIGH-CARDINALITY — use label encoding (integer codes), NOT get_dummies or drop ***"
            elif dtype == 'object' and n <= 20:
                card_note = " (low-cardinality — ok for get_dummies)"
            stats_lines.append(f"COLUMN {col}: dtype={dtype}, unique={n}, nulls={nulls}{card_note}")
        
        raw_stats = "\n".join(stats_lines)
        print(f" Raw stats collected ({len(stats_lines)} lines)")
        
    except Exception as e:
        raw_stats = f"Could not read data: {e}"
        print(f" EDA data read failed: {e}")
    
    task_type = "regression"  # default
    if target_col:
        n_unique = df[target_col].nunique()
        if n_unique <= 10:
            task_type = "classification" if n_unique > 2 else "binary_classification"
    
    eda_rag_context = retrieve_eda_context(competition, task_type, raw_stats)
    
    user_content = f"Competition: {competition}\n\nRaw data statistics:\n{raw_stats}"
    if eda_rag_context:
        user_content += f"\n\n## RETRIEVED EDA KNOWLEDGE (use this to guide your analysis):\n{eda_rag_context}"
    user_content += "\n\nProvide EDA summary with feature engineering ideas and modeling recommendations."
    
    response = eda_llm.invoke([
        {"role": "system", "content": eda_system_prompt},
        {"role": "user", "content": user_content}
    ])
    
    eda_summary = re.sub(r"<think>.*?</think>", "", response.content, flags=re.DOTALL).strip()
    print(f" EDA Summary generated ({len(eda_summary)} chars)")
    
    return {
        "messages": [AIMessage(content=f"EDA complete for {competition}")],
        "next": "supervisor",
        "eda_summary": eda_summary,
        "eda_rag_context": eda_rag_context,
        "iteration_count": state.get("iteration_count", 0) + 1
    }

print(" EDA Agent ready (with RAG retrieval: competition facts, policies, pitfalls)")

✅ EDA Agent ready (with RAG retrieval: competition facts, policies, pitfalls)


## Feature Engineering Agent

In [22]:
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq
import pandas as pd
import numpy as np
import re

feat_eng_llm = ChatGroq(model="qwen/qwen3-32b", temperature=0.3, max_tokens=4096)

feat_eng_system_prompt = """
You are a Feature Engineering Specialist Agent for a Kaggle regression competition.

You receive: raw data sample, EDA insights, and previous score (if retry).

Your job: generate a Python code snippet with ONLY feature engineering functions.
The coder agent will integrate your features into the solution.

Output format:
```python
def engineer_features(df):
    return df
```

RULES:
- Only use pandas and numpy
- Handle NaN values (use fillna)
- Create interaction features (ratios, products)
- Create binned features from continuous variables
- Extract datetime features from any date/datetime columns
- Do NOT drop columns, only ADD new ones
- Do NOT import sklearn or fit any models
- Be creative: log transforms, polynomial features, geographic clustering
- NEVER create features derived from the target column (no is_zero_target, no target bins, no target mean encoding).
  These cause data leakage — the feature exists in train but not in test, destroying model generalization.
"""

def feature_engineer_node(state: AgentState) -> dict:
    dataset_paths = state.get("dataset_paths", {})
    train_path = dataset_paths.get("train", "")
    eda_summary = state.get("eda_summary", "")
    eval_score = state.get("eval_score")
    prev_code = state.get("solution_code", "")
    
    print(f"\n Feature Engineer Agent")
    
    data_info = ""
    try:
        df = pd.read_csv(train_path, nrows=50)
        data_info = f"Columns: {list(df.columns)}\n"
        data_info += f"Dtypes:\n{df.dtypes.to_string()}\n"
        data_info += f"Sample (3 rows):\n{df.head(3).to_string()}\n"
        data_info += f"Describe:\n{df.describe().to_string()}\n"
        
        numeric = df.select_dtypes(include=[np.number])
        if 'target' in numeric.columns:
            corr = numeric.corr()['target'].drop('target').sort_values(key=abs, ascending=False)
            data_info += f"\nCorrelations with target:\n{corr.to_string()}"
    except Exception as e:
        data_info = f"Could not read data: {e}"
    
    user_msg = f"Competition data:\n{data_info}"
    if eda_summary:
        user_msg += f"\n\nEDA Insights:\n{eda_summary[:1500]}"
    if eval_score:
        user_msg += f"\n\nPrevious MSE score: {eval_score:.2f} (needs improvement!)"
        user_msg += f"\nPrevious approach had RMSE {np.sqrt(eval_score):.2f} - create features to beat this."
    
    response = feat_eng_llm.invoke([
        {"role": "system", "content": feat_eng_system_prompt},
        {"role": "user", "content": user_msg}
    ])
    
    feat_code = re.sub(r'<think>.*?</think>', '', response.content, flags=re.DOTALL).strip()
    
    print(f" Feature engineering suggestions generated ({len(feat_code)} chars)")
    
    code_match = re.search(r"```python\n(.*?)\n```", feat_code, re.DOTALL)
    feat_snippet = code_match.group(1) if code_match else feat_code[:2000]
    
    return {
        "messages": [AIMessage(content=feat_snippet)],
        "next": "supervisor",
        "feature_eng_code": feat_snippet[:2000],
        "iteration_count": state.get("iteration_count", 0) + 1
    }

print(" Feature Engineer Agent ready")


✅ Feature Engineer Agent ready


## Executor Agent

In [23]:
import ast, subprocess, re, sys, importlib, inspect
from langchain_core.messages import AIMessage
from pathlib import Path

def is_valid_python(code: str) -> tuple:
    try:
        ast.parse(code)
        return True, None
    except SyntaxError as e:
        return False, f"SyntaxError at line {e.lineno}: {e.msg}"


def get_valid_params(class_name: str, module_hint: str = "sklearn") -> set:
    """Dynamically discover valid constructor params for ANY sklearn class by introspecting the actual class."""
    search_modules = [
        "sklearn.ensemble", "sklearn.linear_model", "sklearn.tree",
        "sklearn.svm", "sklearn.neighbors", "sklearn.naive_bayes",
        "sklearn.neural_network", "sklearn.cluster", "sklearn.decomposition",
        "sklearn.impute", "sklearn.preprocessing",
    ]
    for mod_path in search_modules:
        try:
            mod = importlib.import_module(mod_path)
            cls = getattr(mod, class_name, None)
            if cls is not None:
                sig = inspect.signature(cls.__init__)
                params = {p for p in sig.parameters if p != "self"}
                return params
        except Exception:
            continue
    return set()  # unknown class — don't touch it


def sanitize_sklearn_params(code: str) -> tuple:
    """Remove invalid sklearn constructor kwargs by introspecting the actual class at runtime.
    No hardcoded param lists — discovers valid params dynamically from sklearn."""
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return code, []
    
    fixes = []
    for node in ast.walk(tree):
        if not isinstance(node, ast.Call):
            continue
        func_name = None
        if isinstance(node.func, ast.Name):
            func_name = node.func.id
        elif isinstance(node.func, ast.Attribute):
            func_name = node.func.attr
        if not func_name:
            continue
        
        valid = get_valid_params(func_name)
        if not valid:
            continue  # unknown class, skip
        
        for kw in node.keywords:
            if kw.arg is not None and kw.arg not in valid:
                code = re.sub(r',\s*' + kw.arg + r'\s*=[^,)]+', '', code)
                code = re.sub(kw.arg + r'\s*=[^,)]+\s*,\s*', '', code)
                fixes.append(f"Removed invalid param '{kw.arg}' from {func_name} (valid: {sorted(valid)[:5]}...)")
    
    return code, fixes


def run_solution_code(solution_path: str) -> dict:
    """Run solution directly as a subprocess — gives real tracebacks with correct line numbers."""
    try:
        solution_dir = Path(solution_path).parent
        print(f"\U0001f3c3 Running solution: {solution_path}")
        
        
        import os as _os
        env = _os.environ.copy()
        env['TOKENIZERS_PARALLELISM'] = 'false'
        env['PYTHONWARNINGS'] = 'ignore::RuntimeWarning'
        
        result = subprocess.run(
            [sys.executable, str(solution_path)],
            cwd=str(solution_dir),
            capture_output=True, text=True, timeout=360, env=env
        )
        
        if result.returncode != 0:
            filtered_lines = []
            for line in result.stderr.splitlines():
                if 'RuntimeWarning' in line: continue
                if 'messagestream.MessageStream' in line: continue
                if '<frozen importlib._bootstrap>' in line: continue
                if 'tokenizers' in line.lower(): continue
                if line.count('DType') > 3: continue  # BoolDType/dtype spam
                if line.count('<class') > 3: continue  # class listing spam
                filtered_lines.append(line)
            filtered_stderr = '\n'.join(filtered_lines)
            error_msg = f"Execution Error:\n{filtered_stderr[-4000:]}"
            print(f"\u274c {error_msg[:500]}")
            return {"success": False, "error": error_msg, "submission_path": None, "stdout": result.stdout}
        
        sub_paths = [
            Path(WORK_DIR) / "submission.csv",
            solution_dir / "submission.csv",
        ]
        for sp in sub_paths:
            if sp.exists():
                import pandas as _pd
                sub = _pd.read_csv(sp)
                print(f"\u2705 Submission created: {sp} (shape={sub.shape})")
                if 'prediction' in sub.columns:
                    print(f"   Pred range: {sub['prediction'].min():.2f} to {sub['prediction'].max():.2f}")
                return {"success": True, "submission_path": str(sp.resolve()), "error": None, "stdout": result.stdout}
        
        return {"success": False, "error": "submission.csv not created\n" + result.stdout[-500:], "submission_path": None, "stdout": result.stdout}
    
    except subprocess.TimeoutExpired:
        return {"success": False, "error": "Execution timeout (5 min limit)", "submission_path": None, "stdout": ""}
    except Exception as e:
        return {"success": False, "error": str(e), "submission_path": None, "stdout": ""}


def enrich_error_with_data_context(error_msg: str, state: dict) -> str:
    """When code fails, read the actual data files and append column info to the error.
    This gives the reviewer/coder the ground truth about what columns exist."""
    try:
        import pandas as pd
        dataset_paths = state.get("dataset_paths", {})
        train_path = dataset_paths.get("train", "")
        if not train_path or not Path(train_path).exists():
            return error_msg
        
        df = pd.read_csv(train_path, nrows=5)
        context = "\n\n--- ACTUAL DATA CONTEXT (from train.csv) ---"
        context += f"\nColumns ({len(df.columns)}): {list(df.columns)}"
        context += f"\nDtypes:\n{df.dtypes.to_string()}"
        context += f"\nSample row:\n{df.head(1).to_string()}"
        context += "\n--- END DATA CONTEXT ---"
        return error_msg + context
    except Exception:
        return error_msg


def _extract_cv_rmse(stdout: str) -> float:
    """Extract CV score from solution output. Only matches MSE/RMSE patterns, ignores accuracy/AUC.
    
    Handles both raw-scale scores (MSE ~10000+) and log-scale scores (MSE ~0.5-5.0).
    """
    import re as _re
    best_rmse = float('inf')
    for line in stdout.split('\n'):
        line_upper = line.upper()
        if any(skip in line_upper for skip in ['ACCURACY', 'AUC', 'F1', 'PRECISION', 'RECALL']):
            continue
        if 'MSE' in line_upper and ('CV' in line_upper or 'CROSS' in line_upper or 'VALIDATION' in line_upper):
            nums = _re.findall(r'[\d]+\.[\d]+', line)
            for n in nums:
                val = float(n)
                if 0.001 < val < 1_000_000:  # FIX: accept log-scale MSE (e.g. 0.83) and raw MSE
                    rmse_val = val ** 0.5  # Convert MSE to RMSE for comparison
                    if rmse_val < best_rmse:
                        best_rmse = rmse_val
        elif 'RMSE' in line_upper:
            nums = _re.findall(r'[\d]+\.[\d]+', line)
            for n in nums:
                val = float(n)
                if 0.01 < val < 10_000 and val < best_rmse:  # FIX: accept small RMSE values
                    best_rmse = val
        elif 'MSE' in line_upper and 'RMSE' not in line_upper:
            nums = _re.findall(r'[\d]+\.[\d]+', line)
            for n in nums:
                val = float(n)
                if 0.001 < val < 1_000_000:
                    rmse_val = val ** 0.5
                    if rmse_val < best_rmse:
                        best_rmse = rmse_val
    return best_rmse


def _run_single_candidate(candidate_path: str, state: dict) -> dict:
    """Run a single candidate: auto-fix common issues, sanitize params, validate syntax, execute."""
    try:
        _path_ok, _path_err = InputValidator.validate_llm_output_as_path(candidate_path, WORK_DIR)
        if not _path_ok:
            monitor.log_event("code_executor", "path_traversal_blocked", _path_err, "critical")
            return {"success": False, "error": f"PATH BLOCKED: {_path_err}", "cv_rmse": float('inf')}

        sub_path = Path(WORK_DIR) / "submission.csv"
        if sub_path.exists():
            sub_path.unlink()
        
        with open(candidate_path, "r", encoding="utf-8") as f:
            code = f.read()
        
        if ', squared=False' in code:
            code = code.replace(', squared=False', '')
            print(f"  \U0001f527 Auto-fixed: removed squared=False (sklearn compat)")
        if 'squared=False' in code:
            code = code.replace('squared=False', '')
            print(f"  \U0001f527 Auto-fixed: removed squared=False (sklearn compat)")
        
        code, fixes = sanitize_sklearn_params(code)
        if fixes:
            for fix in fixes:
                print(f"  \U0001f527 {fix}")
        
        with open(candidate_path, "w", encoding="utf-8") as f:
            f.write(code)
        
        is_valid, err = is_valid_python(code)
        if not is_valid:
            return {"success": False, "error": f"Syntax: {err}", "cv_rmse": float('inf')}
        
        is_safe, guard_issues, guard_severity = CodeGuardrails.check_code(code, WORK_DIR)
        if not is_safe:
            monitor.track_guardrail_violation("code_executor", guard_issues)
            return {"success": False, "error": f"BLOCKED by guardrails: {'; '.join(guard_issues)}", "cv_rmse": float('inf')}
        if guard_issues:  # warnings
            for gi in guard_issues:
                monitor.log_event("code_executor", "guardrail_warning", gi, "warning")
        
        result = run_solution_code(candidate_path)
        cv_rmse = _extract_cv_rmse(result.get("stdout", ""))
        result["cv_rmse"] = cv_rmse
        
        sub_path = Path(WORK_DIR) / "submission.csv"
        if result["success"] and sub_path.exists():
            candidate_name = Path(candidate_path).stem
            unique_sub = Path(WORK_DIR) / f"submission_{candidate_name}.csv"
            import shutil
            shutil.copy2(sub_path, unique_sub)
            result["submission_path"] = str(unique_sub.resolve())
        
        return result
    except Exception as e:
        return {"success": False, "error": str(e), "cv_rmse": float('inf')}


def code_executor_node(state: AgentState) -> dict:
    """Runs ALL candidate solutions, picks the best by CV RMSE score.
    Tracks best-ever solution to prevent regression across iterations."""
    candidate_paths = state.get("candidate_paths", [])
    solution_path = state.get("solution_path", "")
    best_eval_score = state.get("best_eval_score")
    best_solution_code = state.get("best_solution_code")
    best_submission_path = state.get("best_submission_path")
    
    for old_sub in [Path(WORK_DIR) / "submission.csv"]:
        if old_sub.exists():
            old_sub.unlink()
    
    all_paths = []
    if candidate_paths:
        all_paths = [p for p in candidate_paths if Path(p).exists()]
    if not all_paths and solution_path and Path(solution_path).exists():
        all_paths = [solution_path]
    
    if not all_paths:
        return {
            "messages": [AIMessage(content="No solution file found to execute")],
            "next": "supervisor",
            "code_validated": False,
            "iteration_count": state.get("iteration_count", 0) + 1
        }
    
    print(f"\n\u2699\ufe0f Code Executor Agent: testing {len(all_paths)} candidate(s)")
    if best_eval_score is not None:
        print(f"   Best-ever MSE so far: {best_eval_score:.2f}")
    
    results = []
    for i, path in enumerate(all_paths):
        print(f"\n  \U0001f4cb Candidate {i+1}/{len(all_paths)}: {Path(path).name}")
        result = _run_single_candidate(path, state)
        result["path"] = path
        results.append(result)
        status = f"CV RMSE={result['cv_rmse']:.2f}" if result["success"] else f"FAILED: {result.get('error', '')[:100]}"
        icon = "\u2705" if result["success"] else "\u274c"
        print(f"  {icon} {status}")
    
    successful = [r for r in results if r["success"]]
    
    if successful:
        best = min(successful, key=lambda r: r["cv_rmse"])
        best_path = best["path"]
        
        import shutil
        best_sub = best.get("submission_path", "")
        main_sub = Path(WORK_DIR) / "submission.csv"
        if best_sub and Path(best_sub).exists():
            shutil.copy2(best_sub, main_sub)
            print(f"\n\U0001f3c6 Best candidate: {Path(best_path).name} (CV RMSE={best['cv_rmse']:.2f})")
        else:
            print(f"\n\U0001f3c6 Best: {Path(best_path).name} (CV RMSE={best['cv_rmse']:.2f})")
        
        if best_path != solution_path and solution_path:
            shutil.copy2(best_path, solution_path)
        
        with open(best_path, "r") as f:
            best_code = f.read()
        
        output_msg = f"Tested {len(all_paths)} candidates. Best: CV RMSE={best['cv_rmse']:.2f}"
        output_msg += f"\nSubmission at: {best.get('submission_path', 'N/A')}"
        if len(successful) > 1:
            scores = sorted([r['cv_rmse'] for r in successful])
            output_msg += f"\nAll CV scores: {[f'{s:.2f}' for s in scores]}"
        if best.get("stdout"):
            output_msg += f"\nOutput: {best['stdout'][:300]}"
        
        return {
            "messages": [AIMessage(content=output_msg)],
            "next": "supervisor",
            "code_validated": True,
            "solution_code": best_code,
            "solution_path": solution_path or best_path,
            "cv_score": f"CV RMSE={best['cv_rmse']:.2f}" if best['cv_rmse'] != float('inf') else None,
            "submission_path": best.get("submission_path"),
            "validation_errors": None,
            "candidate_paths": [],  # Clear so next cycle doesn't re-run stale candidates
            "eval_passed": None,  # Reset: new submission needs evaluation
            "iteration_count": state.get("iteration_count", 0) + 1
        }
    else:
        all_errors = []
        for i, r in enumerate(results):
            err = r.get("error", "Unknown error")
            lines = err.split('\n')
            tb_start = 0
            for j, line in enumerate(lines):
                if 'Traceback' in line or 'STAGE_FAILED' in line:
                    tb_start = j
            relevant = '\n'.join(lines[tb_start:])[-500:]
            all_errors.append(f"--- Candidate {i+1} ---\n{relevant}")
        
        combined = '\n'.join(all_errors)
        enriched = enrich_error_with_data_context(combined, state)
        
        return {
            "messages": [AIMessage(content=f"All {len(results)} candidates FAILED:\n{enriched[:3000]}")],
            "next": "supervisor",
            "code_validated": False,
            "validation_errors": enriched[:4000],
            "error_history": [f"ALL {len(results)} CANDIDATES FAILED:\n{combined[:1000]}"],
            "candidate_paths": [],  # Clear stale candidates
            "iteration_count": state.get("iteration_count", 0) + 1
        }

print("\u2705 Code Executor Agent ready (fixed CV extraction + best-ever tracking)")

✅ Code Executor Agent ready (fixed CV extraction + best-ever tracking)


## Evaluator Agent

In [24]:
import os
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_core.callbacks import BaseCallbackHandler
from langchain_groq import ChatGroq
import pandas as pd
import numpy as np
import re, json

class EvaluatorTracer(BaseCallbackHandler):
    def __init__(self):
        self.trace_lines = []

    def reset(self):
        self.trace_lines = []

    def on_llm_start(self, serialized, prompts=None, messages=None, **kwargs):
        model = serialized.get("kwargs", {}).get("model_name", serialized.get("id", ["?"])[-1])
        self.trace_lines.append(f"  [LLM START] Calling {model}")
        if messages:
            for batch in messages:
                for msg in batch:
                    role = msg.type if hasattr(msg, "type") else "unknown"
                    content = msg.content if hasattr(msg, "content") else str(msg)
                    preview = content[:200].replace("\n", " ")
                    self.trace_lines.append(f"    >> {role}: {preview}{'...' if len(content) > 200 else ''}")

    def on_llm_end(self, response, **kwargs):
        for gen in response.generations:
            for g in gen:
                msg = g.message if hasattr(g, "message") else None
                if not msg:
                    continue
                if msg.content:
                    preview = msg.content[:300].replace("\n", " ")
                    self.trace_lines.append(f"  [LLM THINKING] {preview}{'...' if len(msg.content) > 300 else ''}")
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    names = [tc["name"] for tc in msg.tool_calls]
                    self.trace_lines.append(f"  [LLM DECISION] Selected tools: {names}")
                    for tc in msg.tool_calls:
                        args_str = json.dumps(tc["args"], ensure_ascii=False)[:200]
                        self.trace_lines.append(f"    -> {tc['name']}({args_str})")
                elif not msg.content:
                    self.trace_lines.append(f"  [LLM END] (empty response)")

    def get_trace(self):
        return "\n".join(self.trace_lines)

tracer = EvaluatorTracer()
tracer_config = {"callbacks": [tracer]}

evaluator_llm = ChatGroq(model="qwen/qwen3-32b", temperature=0, max_tokens=4096)

MSE_THRESHOLD = 10000

eval_history: list[dict] = []


@tool
def validate_submission_format(submission_path: str) -> dict:
    """
    Check if a submission CSV has valid format and reasonable predictions.

    Args:
        submission_path: Absolute path to the submission CSV

    Returns:
        dict with: valid (bool), shape, columns, issues (list of problems found)
    """
    df = pd.read_csv(submission_path)
    issues = []
    if 'index' not in df.columns:
        issues.append("Missing 'index' column")
    if 'prediction' not in df.columns:
        issues.append("Missing 'prediction' column")
    if df['prediction'].isna().any():
        issues.append(f"{df['prediction'].isna().sum()} NaN predictions")
    if (df['prediction'] < 0).any():
        issues.append(f"{(df['prediction'] < 0).sum()} negative predictions")
    return {
        "valid": len(issues) == 0,
        "shape": list(df.shape),
        "columns": list(df.columns),
        "issues": issues,
        "prediction_range": [round(float(df['prediction'].min()), 2), round(float(df['prediction'].max()), 2)],
    }


@tool
def score_submission(submission_path: str, solution_path: str) -> dict:
    """
    Compute MSE by comparing a submission CSV against the ground-truth solution CSV.

    Args:
        submission_path: Absolute path to the submission CSV (must have 'index' and 'prediction' columns)
        solution_path: Absolute path to the solution CSV (must have 'index', 'prediction', optionally 'Usage' columns)

    Returns:
        dict with: mse, rmse, matched_rows, residual_stats, error_by_bin
    """
    from sklearn.metrics import mean_squared_error
    from pathlib import Path

    if not Path(submission_path).exists():
        return {"error": f"Submission file not found: {submission_path}"}
    if not Path(solution_path).exists():
        return {"error": f"Solution file not found: {solution_path}"}

    sub_df = pd.read_csv(submission_path)
    sol_df = pd.read_csv(solution_path)

    usage_filter = None
    if 'Usage' in sol_df.columns:
        total_rows = len(sol_df)
        sol_df = sol_df[sol_df['Usage'] == 'Public']
        usage_filter = f"Filtered to {len(sol_df)}/{total_rows} Public rows (Private rows excluded per Kaggle rules)"

    merged = sol_df.merge(sub_df, on='index', suffixes=('_true', '_pred'))
    if len(merged) == 0:
        return {"error": "No matching rows (index mismatch)"}

    mse = float(mean_squared_error(merged['prediction_true'], merged['prediction_pred']))
    rmse = float(np.sqrt(mse))
    residuals = merged['prediction_true'] - merged['prediction_pred']

    error_by_bin = {}
    try:
        merged['abs_error'] = np.abs(residuals)
        merged['target_bin'] = pd.qcut(merged['prediction_true'], q=4, duplicates='drop')
        for bin_name, group in merged.groupby('target_bin'):
            error_by_bin[str(bin_name)] = round(float(group['abs_error'].mean()), 2)
    except Exception:
        pass

    result = {
        "mse": round(mse, 2),
        "rmse": round(rmse, 2),
        "matched_rows": len(merged),
        "residual_stats": {
            "mean": round(float(residuals.mean()), 2),
            "std": round(float(residuals.std()), 2),
            "worst_over": int((residuals < -200).sum()),
            "worst_under": int((residuals > 200).sum()),
        },
        "error_by_bin": error_by_bin,
    }
    if usage_filter:
        result["usage_note"] = usage_filter
    return result



def _condense_eda(eda_summary: str) -> str:
    """Extract key stats from EDA summary into ~200 char digest for evaluator."""
    if not eda_summary:
        return ""
    import re as _re
    lines = eda_summary.lower()
    parts = []
    z = _re.search(r"(\d+\.?\d*)%\s*zero", lines) or _re.search(r"zero.*?(\d+\.?\d*)%", lines)
    if z:
        parts.append(f"target {z.group(1)}% zeros")
    r = _re.search(r"range.*?(\d+\.?\d*).*?to.*?(\d+\.?\d*)", lines)
    if r:
        parts.append(f"range {r.group(1)}-{r.group(2)}")
    hc = _re.findall(r"high.card(?:inality)?[^:]*:\s*([\w_,\s]+)", lines)
    if hc:
        parts.append(f"high-card: {hc[0].strip()[:80]}")
    sk = _re.findall(r"skew(?:ed)?[^:]*:\s*([\w_,\s]+)", lines)
    if sk:
        parts.append(f"skewed: {sk[0].strip()[:60]}")
    ms = _re.findall(r"(\w+).*?missing.*?(\d+)", lines[:500])
    if ms:
        parts.append(f"missing: {', '.join(f'{n}({c})' for n,c in ms[:3])}")
    if not parts:
        return eda_summary[:200]
    return ". ".join(parts)



def retrieve_eval_strategies(
    mse: float,
    prev_mse: float,
    residual_stats: dict,
    error_by_bin: dict,
    eda_summary: str = ""
) -> str:
    """Retrieve relevant improvement strategies from RAG based on evaluation signals."""
    query_parts = []
    is_plateau = False

    if prev_mse is not None and prev_mse > 0:
        pct_change = abs(mse - prev_mse) / prev_mse * 100
        if pct_change < 2:
            is_plateau = True
            query_parts.append("plateau model stalled hyperparameters exhausted structural change needed")
        else:
            direction = "improved" if mse < prev_mse else "regressed"
            query_parts.append(f"MSE {direction} by {pct_change:.1f}%")

    if residual_stats:
        mean_res = residual_stats.get("mean", 0)
        if mean_res > 10:
            query_parts.append(f"systematic underprediction bias mean residual {mean_res:.1f}")
        elif mean_res < -10:
            query_parts.append(f"systematic overprediction bias mean residual {mean_res:.1f}")
        worst_under = residual_stats.get("worst_under", 0)
        if worst_under > 500:
            query_parts.append(f"clipping range worst_under {worst_under} underprediction")

    if error_by_bin:
        bin_errors = list(error_by_bin.values())
        if bin_errors:
            max_err = max(bin_errors)
            min_err = min(bin_errors) if min(bin_errors) > 0 else 1
            if max_err > 2 * min_err:
                worst_bin = max(error_by_bin, key=error_by_bin.get)
                query_parts.append(f"error bin imbalance worst bin {worst_bin} targeted feature engineering")

    if not query_parts:
        query_parts.append("general regression improvement strategy")

    query = " ".join(query_parts)
    print(f"  \U0001f4ca Eval strategy RAG query: {query[:120]}")

    try:
        docs = eda_store.similarity_search(
            query,
            k=EVAL_RAG_TOP_K,
            filter={"doc_type": "eval_strategy"}
        )
    except Exception as e:
        print(f"  \u26a0\ufe0f Eval strategy retrieval failed: {e}")
        return ""

    if not docs:
        print("  \U0001f4ca No eval strategies found in RAG")
        return ""

    strategies = [doc.page_content for doc in docs]
    print(f"  \U0001f4ca Retrieved {len(docs)} eval strategies")

    retrieved_text = "### RETRIEVED IMPROVEMENT STRATEGIES:\n" + "\n---\n".join(strategies)

    if is_plateau:
        retrieved_text = (
            "**PLATEAU DETECTED**: MSE change < 2%. Hyperparameter tweaks are EXHAUSTED. "
            "Suggest ONLY structural/architectural changes from the retrieved strategies below. "
            "Do NOT suggest any hyperparameter changes.\n\n" + retrieved_text
        )

    return retrieved_text

evaluator_tools = [validate_submission_format, score_submission]

evaluator_system_prompt = f"""You are an Evaluation Agent for a Kaggle competition.
You have two tools. Call them in order:
1. validate_submission_format — check the submission file format
2. score_submission — compute the MSE score against the ground truth

After receiving the score, analyze the results and suggest improvements.
PASS/FAIL rule: MSE <= {MSE_THRESHOLD} is PASS, MSE > {MSE_THRESHOLD} is FAIL.
Use the exact paths from the user message.

IMPORTANT: Do NOT suggest log-transforming the target variable. This causes scale mismatch between CV and evaluation MSE.

When asked to suggest improvements, use the retrieved improvement strategies provided in the analysis prompt.
Base your suggestions on the specific evaluation signals (MSE trend, residual stats, error_by_bin).
Give exact numbers and parameter names. NEVER say vague things without specifics.
Do NOT repeat the same suggestion if the score has not changed."""


def evaluator_node(state: AgentState) -> dict:
    submission_path = state.get("submission_path", "")
    dataset_paths = state.get("dataset_paths", {})
    competition = state.get("competition_name", "unknown")
    prev_score = state.get("eval_score")
    prev_improvements = state.get("eval_improvements", []) or []
    iteration = state.get("iteration_count", 0)
    solution_file = dataset_paths.get("solution", "")
    eda_summary = state.get("eda_summary", "")
    
    best_eval_score = state.get("best_eval_score")
    best_solution_code = state.get("best_solution_code")
    best_submission_path = state.get("best_submission_path")

    tracer.reset()
    trace_lines = []

    user_input = (
        f"Evaluate this submission.\n"
        f"SUBMISSION_PATH: {submission_path}\n"
        f"SOLUTION_PATH: {solution_file}\n"
        f"MSE threshold: {MSE_THRESHOLD}\n"
    )
    if prev_score is not None:
        user_input += f"Previous best MSE: {prev_score:.2f}\n"

    if eda_summary:
        user_input += f"\nEDA summary of training data:\n{_condense_eda(eda_summary)}\n"

    tool_registry = {t.name: t for t in evaluator_tools}
    llm_with_tools = evaluator_llm.bind_tools(evaluator_tools)
    messages = [SystemMessage(content=evaluator_system_prompt), HumanMessage(content=user_input)]

    actual_mse = None
    score_result = None
    format_result = None

    for turn in range(3):
        response = llm_with_tools.invoke(messages, config=tracer_config)

        if not response.tool_calls:
            break

        messages.append(response)

        for tc in response.tool_calls:
            trace_lines.append(f"  [TOOL CALL] {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)[:200]})")
            result = tool_registry[tc["name"]].invoke(tc["args"])
            trace_lines.append(f"  [TOOL RESULT] {tc['name']} -> {json.dumps(result, ensure_ascii=False)[:300]}")
            messages.append(ToolMessage(content=json.dumps(result), tool_call_id=tc["id"]))

            if tc["name"] == "score_submission" and "mse" in result:
                actual_mse = result["mse"]
                score_result = result
            if tc["name"] == "validate_submission_format":
                format_result = result

        if actual_mse is not None:
            break

    if actual_mse is not None:
        mse_score = actual_mse
        passed = mse_score <= MSE_THRESHOLD
    else:
        mse_score = 99999.0
        passed = False
        trace_lines.append("  [ERROR] No MSE score obtained from score_submission")

    analysis = f"MSE={mse_score:.2f} ({'PASS' if passed else 'FAIL'}, threshold={MSE_THRESHOLD})"
    improvements = []

    if actual_mse is not None:
        retrieved_strategies = ""
        if score_result:
            retrieved_strategies = retrieve_eval_strategies(
                mse=mse_score,
                prev_mse=prev_score,
                residual_stats=score_result.get("residual_stats"),
                error_by_bin=score_result.get("error_by_bin"),
                eda_summary=eda_summary,
            )

        analysis_prompt = (
            f"The submission scored MSE={mse_score:.2f} (threshold={MSE_THRESHOLD}). "
            f"{'PASS' if passed else 'FAIL'}. "
        )
        if prev_score is not None:
            direction = "improved" if mse_score < prev_score else "regressed"
            analysis_prompt += f"Previous MSE was {prev_score:.2f} ({direction}). "
        if prev_improvements:
            analysis_prompt += f"Previous suggestions were: {prev_improvements}. Consider whether to refine, repeat, or try different approaches. "
        if retrieved_strategies:
            analysis_prompt += f"\n\n{retrieved_strategies}\n\n"
            analysis_prompt += "Based on the retrieved strategies above, list 3-4 specific improvements as a JSON array of strings. Respond with ONLY the JSON array."
        else:
            analysis_prompt += "List 3-4 specific improvements as a JSON array of strings. Respond with ONLY the JSON array."

        try:
            analysis_resp = evaluator_llm.invoke(messages + [HumanMessage(content=analysis_prompt)])
            resp_text = re.sub(r'<think>.*?</think>', '', analysis_resp.content, flags=re.DOTALL).strip()
            arr_match = re.search(r'\[.*\]', resp_text, re.DOTALL)
            if arr_match:
                improvements = json.loads(arr_match.group())
            if resp_text and not improvements:
                analysis = resp_text[:500]
        except Exception:
            pass

    is_regression = prev_score is not None and mse_score > prev_score

    trace_lines.append(f"  {'PASS' if passed else 'FAIL'} — MSE: {mse_score:.2f} (threshold: {MSE_THRESHOLD})")
    if is_regression:
        trace_lines.append(f"   REGRESSION: {prev_score:.2f} -> {mse_score:.2f}")

    eval_history.append({"iteration": iteration, "mse": mse_score, "passed": passed})
    if len(eval_history) > 1:
        scores = [e["mse"] for e in eval_history]
        trace_lines.append(f"   History: {' -> '.join(f'{s:.0f}' for s in scores)} (best: {min(scores):.0f})")

    new_best_score = best_eval_score
    new_best_code = best_solution_code
    new_best_sub = best_submission_path
    
    if best_eval_score is None or mse_score < best_eval_score:
        new_best_score = mse_score
        new_best_code = state.get("solution_code")
        new_best_sub = submission_path
        trace_lines.append(f"   NEW BEST! MSE: {mse_score:.2f} (previous best: {best_eval_score if best_eval_score else 'none'})")
        try:
            import shutil
            from pathlib import Path
            if submission_path and Path(submission_path).exists():
                best_backup = Path(WORK_DIR) / "submission_best_ever.csv"
                shutil.copy2(submission_path, best_backup)
                new_best_sub = str(best_backup.resolve())
        except Exception:
            pass
    elif is_regression:
        trace_lines.append(f"   Keeping best-ever MSE: {best_eval_score:.2f} (current: {mse_score:.2f} is worse)")

    full_trace = tracer.get_trace() + "\n" + "\n".join(trace_lines)

    return {
        "messages": [AIMessage(content=analysis)],
        "next": "supervisor",
        "eval_score": mse_score,
        "eval_passed": passed,
        "eval_improvements": improvements,
        "eval_error_by_bin": score_result.get("error_by_bin") if score_result else None,
        "eval_trace": full_trace,
        "iteration_count": iteration + 1,
        "best_eval_score": new_best_score,
        "best_solution_code": new_best_code,
        "best_submission_path": new_best_sub,
    }

print("Evaluator Agent ready (deterministic scoring, no hallucination possible)")

Evaluator Agent ready (deterministic scoring, no hallucination possible)


## Supervisor

In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq
import json as _json, re
from pathlib import Path

supervisor_llm = ChatGroq(model="qwen/qwen3-32b", temperature=0, max_tokens=4096)

supervisor_system_prompt = """You are the Supervisor agent in a multi-agent Kaggle competition system.
You decide which agent to call next based on the current workflow state.

- kaggle_agent: Downloads competition files OR uploads submission to Kaggle
- eda_agent: Analyzes training data, produces EDA summary with insights
- feature_engineer_agent: Creates advanced feature engineering code based on data analysis
- coder_agent: Generates Python solution code for the competition
- code_executor_agent: Validates syntax and runs the solution code
- evaluator_agent: Evaluates submission quality by computing MSE score
- END: Finish the workflow

1. Data must be downloaded before analysis (kaggle_agent first if no dataset)
2. EDA must be done before coding (eda_agent before coder_agent)
3. After coder generates code, execute it immediately
4. After execution produces a submission, evaluate it
5. If evaluation PASSES (eval_passed=True) → kaggle_agent to upload → END
6. If evaluation FAILS → route to coder_agent with feedback to improve
7. After successful upload → END

- After coder_agent generates code → ALWAYS route to code_executor_agent next
- After code_executor_agent succeeds → ALWAYS route to evaluator_agent next
- After code_executor_agent fails → ALWAYS route to coder_agent to fix errors
- NEVER call coder_agent without EDA being done first
- NEVER instruct any agent to create features derived from the target column (no is_zero_target, no zero-indicator for target, no target encoding). This is data leakage.
- feature_engineer_agent should only be called ONCE per workflow

- NEVER call the same agent more than 3 times consecutively
- After 3 failed evaluation cycles, route to END
- If iteration count > 40, route to END

- When routing to coder_agent, include ALL column details from EDA in your instruction
- The coder has ZERO knowledge about the competition — everything must come from your instruction
- If code execution fails, include the EXACT error message when routing to coder_agent
- The coder can use sklearn, xgboost, and lightgbm. Suggest LGBMRegressor or XGBRegressor for best accuracy on tabular data
- For categorical columns, tell the coder to use LABEL ENCODING (integer codes) for tree-based models, NOT target encoding

{state_summary}

{{"next_agent": "agent_name_or_END", "instruction": "detailed instruction for the agent"}}
"""

MAX_ITERATIONS = 60


def _parse_supervisor_json(raw_content: str, llm) -> dict:
    clean = re.sub(r'<think>.*?</think>', '', raw_content, flags=re.DOTALL).strip()
    
    for attempt in [
        lambda: _json.loads(clean),
        lambda: _json.loads(re.search(r'```(?:json)?\s*(\{.*?\})\s*```', clean, re.DOTALL).group(1)),
        lambda: _json.loads(re.search(r'\{[^}]+\}', clean).group()),
    ]:
        try:
            return attempt()
        except:
            continue
    
    print("\u26a0\ufe0f Retrying supervisor LLM for valid JSON...")
    retry = llm.invoke([
        {"role": "system", "content": "Respond with ONLY a JSON object. No text, no markdown."},
        {"role": "user", "content": f"Convert to valid JSON with keys next_agent and instruction:\n{clean[:500]}"}
    ])
    retry_clean = re.sub(r'<think>.*?</think>', '', retry.content, flags=re.DOTALL).strip()
    try:
        return _json.loads(retry_clean)
    except:
        pass
    try:
        return _json.loads(re.search(r'\{[^}]+\}', retry_clean).group())
    except:
        pass
    
    return None


def supervisor_node(state: AgentState) -> dict:
    """LLM-driven supervisor with post-decision safety overrides."""
    iteration = state.get("iteration_count", 0)
    
    competition = state.get("competition_name", "unknown")
    dataset_paths = state.get("dataset_paths", {})
    solution_path = state.get("solution_path")
    submission_path = state.get("submission_path")
    code_validated = state.get("code_validated", False)
    submission_uploaded = state.get("submission_uploaded", False)
    eval_passed = state.get("eval_passed")
    eval_score = state.get("eval_score")
    eda_summary = state.get("eda_summary")
    validation_errors = state.get("validation_errors")
    best_eval_score = state.get("best_eval_score")
    feature_eng_done = bool(state.get("feature_eng_code"))
    
    print(f"\n\U0001f441\ufe0f  Supervisor (iteration {iteration})")
    print(f"   Dataset: {list(dataset_paths.keys()) if dataset_paths else 'EMPTY'}")
    print(f"   EDA: {'done' if eda_summary else 'not done'}, Solution: {solution_path or ''}")
    print(f"   Validated: {code_validated}, Submission: {submission_path or ''}")
    print(f"   Eval: score={eval_score}, passed={eval_passed}, best_ever={best_eval_score}")
    print(f"   Uploaded: {submission_uploaded}")
    
    if submission_uploaded:
        print(f"   \u27a1\ufe0f  Next: END (uploaded)")
        return {"messages": [AIMessage(content="Done. Submission uploaded.")], "next": "END", "iteration_count": iteration + 1}
    
    if iteration >= MAX_ITERATIONS:
        if code_validated and submission_path and eval_passed is None:
            return {"messages": [AIMessage(content="Max iterations — evaluating final submission.")], "next": "evaluator_agent", "iteration_count": iteration + 1}
        best_sub = state.get("best_submission_path")
        if best_sub and best_eval_score is not None:
            print(f"    FINAL: Submitting best-ever solution (MSE={best_eval_score:.2f}, threshold={MSE_THRESHOLD})")
            print(f"    Submission file: {best_sub}")
            return {"messages": [AIMessage(content=f"Max iterations reached. Uploading best-ever submission (MSE={best_eval_score:.2f}).")], "next": "kaggle_agent", "iteration_count": iteration + 1,
                    "eval_passed": best_eval_score <= MSE_THRESHOLD, "submission_path": best_sub}
        print(f"   ️ No scored submission available to upload.")
        return {"messages": [AIMessage(content="Max iterations reached. No scored submission.")], "next": "END", "iteration_count": iteration}
    
    state_summary = f"""Competition: {competition}
Dataset downloaded: {bool(dataset_paths)} (files: {list(dataset_paths.keys()) if dataset_paths else 'NONE'})
Train path: {dataset_paths.get('train', 'N/A')}
Test path: {dataset_paths.get('test', 'N/A')}
EDA done: {bool(eda_summary)}
Feature engineering done: {feature_eng_done}
Solution generated: {bool(solution_path)} (path: {solution_path or 'NONE'})
Code validated and executed: {code_validated}
Validation errors: {validation_errors or 'NONE'}
Submission file: {submission_path or 'NONE'}
Evaluation MSE score: {eval_score if eval_score is not None else 'NOT YET EVALUATED'}
Best-ever MSE score: {f'{best_eval_score:.2f}' if best_eval_score is not None else 'NONE'}
Evaluation passed (MSE <= {MSE_THRESHOLD}): {eval_passed if eval_passed is not None else 'NOT YET EVALUATED'}
Submission uploaded to Kaggle: {submission_uploaded}
Iteration: {iteration}"""
    
    if state.get("messages"):
        msg = state["messages"][-1]
        content = msg.content if hasattr(msg, 'content') else str(msg)
        clean_content = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL).strip()
        clean_content = re.sub(r'```python\n.*?\n```', '[code block]', clean_content, flags=re.DOTALL)
        state_summary += f"\nLast agent output: {clean_content[:500]}"
    
    column_info = ""
    if dataset_paths.get('train'):
        try:
            col_info = get_column_info(dataset_paths['train'])
            column_info += f"Column statistics from data:\n{col_info}"
        except:
            pass
    if eda_summary:
        column_info += f"\n\nFull EDA Analysis:\n{eda_summary[:3000]}"
    
    response = supervisor_llm.invoke([
        {"role": "system", "content": supervisor_system_prompt.format(state_summary=state_summary)},
        {"role": "user", "content": f"Column info:\n{column_info}\n\nBased on the current state, which agent should act next?"}
    ])
    
    decision = _parse_supervisor_json(response.content, supervisor_llm)
    if decision is None:
        decision = {"next_agent": "END", "instruction": "Failed to parse routing decision"}
    
    next_agent = decision.get("next_agent", "END")
    instruction = decision.get("instruction", "")
    
    valid_agents = {"kaggle_agent", "eda_agent", "feature_engineer_agent", "coder_agent",
                    "code_executor_agent", "evaluator_agent", "END"}
    agent_aliases = {
        "validator_agent": "code_executor_agent", "validator": "code_executor_agent",
        "executor_agent": "code_executor_agent", "executor": "code_executor_agent",
        "coder": "coder_agent", "eda": "eda_agent", "kaggle": "kaggle_agent",
        "evaluator": "evaluator_agent", "feature_engineer": "feature_engineer_agent",
        "FINISH": "END", "end": "END", "done": "END",
    }
    if next_agent not in valid_agents:
        lower = next_agent.lower()
        matched = next((v for v in valid_agents if v.lower() == lower), None)
        if matched:
            next_agent = matched
        else:
            next_agent = agent_aliases.get(next_agent, agent_aliases.get(lower, "END"))
    
    if next_agent == "coder_agent" and not eda_summary:
        print(f"   \U0001f6d1 Override: coder needs EDA first")
        next_agent = "eda_agent"
        instruction = "Analyze the training data before coding."
    
    if next_agent not in ("coder_agent", "END") and validation_errors and not code_validated:
        print(f"   \U0001f6d1 Override: execution failed, routing to coder")
        next_agent = "coder_agent"
    
    if code_validated and submission_path and eval_passed is None and next_agent not in ("evaluator_agent",):
        print(f"   \U0001f6d1 Override: un-evaluated submission, routing to evaluator")
        next_agent = "evaluator_agent"
        instruction = "Evaluate the latest submission."
    
    if eval_passed and not submission_uploaded and next_agent == "END":
        print(f"   \U0001f6d1 Override: eval passed, must upload first")
        next_agent = "kaggle_agent"
        instruction = "Upload the passing submission to Kaggle."
    
    if next_agent == "END" and best_eval_score is not None and not submission_uploaded:
        best_sub = state.get("best_submission_path")
        if best_sub:
            print(f"    Uploading best-ever submission (MSE={best_eval_score:.2f}) before ending")
            next_agent = "kaggle_agent"
            instruction = f"Upload the best submission (MSE={best_eval_score:.2f}) to Kaggle."
    
    if next_agent == "feature_engineer_agent" and feature_eng_done:
        next_agent = "coder_agent"
        instruction += "\nFeature engineering code is ready — integrate it."
    
    
    if next_agent == "coder_agent":
        if eda_summary:
            instruction += "\n\n--- FULL EDA ANALYSIS ---\n" + eda_summary[:3000]
        if validation_errors:
            instruction += chr(10) + chr(10) + "## EXECUTION ERRORS (fix these):" + chr(10) + validation_errors[:2000]
        if eval_score is not None and not eval_passed:
            instruction += f"\n\n## EVAL FEEDBACK:\nMSE: {eval_score:.0f} (threshold: {MSE_THRESHOLD})"
            if best_eval_score is not None:
                instruction += f"\nBest-ever MSE: {best_eval_score:.0f} — do NOT regress!"
            cv = state.get("cv_score")
            if cv:
                instruction += f"\nCV score: {cv}"
                instruction += "\nWARNING: If CV << actual MSE, your CV uses the wrong scale. Use np.expm1() before scoring."
            eval_imps = state.get("eval_improvements", [])
            if eval_imps:
                instruction += "\nIssues: " + "; ".join(eval_imps)
            eval_bins = state.get("eval_error_by_bin")
            if eval_bins:
                instruction += "\n\n## ERROR BY TARGET BIN (focus feature engineering on worst bins):"
                for bin_range, bin_err in sorted(eval_bins.items(), key=lambda x: x[1], reverse=True):
                    instruction += f"\n  {bin_range}: mean_abs_error={bin_err}"
    
    print(f"   \u27a1\ufe0f  Next: {next_agent}")
    print(f"   \U0001f4dd {instruction[:150]}")
    
    result = {
        "messages": [AIMessage(content=instruction)],
        "next": next_agent,
        "iteration_count": iteration + 1
    }
    if next_agent == "kaggle_agent" and state.get("best_submission_path"):
        result["submission_path"] = state["best_submission_path"]
    return result

print("\u2705 Supervisor ready (LLM-driven + 4 safety invariants)")

✅ Supervisor ready (LLM-driven + 4 safety invariants)


In [26]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)
workflow.add_node("kaggle_agent", monitored_node("kaggle_agent")(kaggle_node))
workflow.add_node("eda_agent", monitored_node("eda_agent")(eda_node))
workflow.add_node("feature_engineer_agent", monitored_node("feature_engineer")(feature_engineer_node))
workflow.add_node("coder_agent", monitored_node("coder_agent")(coder_node))
workflow.add_node("code_executor_agent", monitored_node("code_executor")(code_executor_node))
workflow.add_node("evaluator_agent", monitored_node("evaluator")(evaluator_node))
workflow.add_node("supervisor", monitored_node("supervisor")(supervisor_node))

workflow.set_entry_point("supervisor")
workflow.add_conditional_edges(
    "supervisor",
    lambda state: state.get("next", "END"),
    {
        "kaggle_agent": "kaggle_agent",
        "eda_agent": "eda_agent",
        "feature_engineer_agent": "feature_engineer_agent",
        "coder_agent": "coder_agent",
        "code_executor_agent": "code_executor_agent",
        "evaluator_agent": "evaluator_agent",
        "supervisor": "supervisor",
        "END": END
    }
)
workflow.add_edge("kaggle_agent", "supervisor")
workflow.add_edge("eda_agent", "supervisor")
workflow.add_edge("feature_engineer_agent", "supervisor")
workflow.add_edge("coder_agent", "supervisor")
workflow.add_edge("code_executor_agent", "supervisor")
workflow.add_edge("evaluator_agent", "supervisor")

app = workflow.compile()
app.step_timeout = 600

_user_prompt = f"Solve kaggle competition {COMPETITION_NAME}. Download the data, analyze it with EDA, build a model, evaluate, and submit. Submission format: columns 'index' and 'prediction'."

monitor.start_time = datetime.now()
monitor._alerts.clear()
monitor.logs.clear()
monitor.error_counts.clear()
monitor.metrics.clear()

initial_state = {
    "messages": [HumanMessage(content=_user_prompt)],
    "competition_name": COMPETITION_NAME,
}

print("Starting Workflow")
print(f"Competition: {COMPETITION_NAME}")
print("="*60)

_final_best_score = None
_final_best_sub = None
_final_uploaded = False

for output in app.stream(initial_state, {"recursion_limit": 60}):
    for key, value in output.items():
        print(f"\n{'='*60}")
        print(f"Node: {key}")
        print(f"{'='*60}")
        
        if "eval_trace" in value and value["eval_trace"]:
            print(value["eval_trace"])
        
        if "dataset_paths" in value and value["dataset_paths"]:
            print(f"Dataset: {list(value['dataset_paths'].keys())}")
        
        if "eda_summary" in value and value["eda_summary"]:
            print(f"EDA: {value['eda_summary'][:200]}...")
        
        if "eval_score" in value and value["eval_score"] is not None:
            print(f"Eval MSE: {value['eval_score']:.2f}, Passed: {value.get('eval_passed')}")
        
        if "eval_improvements" in value and value["eval_improvements"]:
            print(f"Evaluator improvements:")
            for imp in value["eval_improvements"]:
                print(f"   - {imp}")
        
        if "solution_path" in value and value["solution_path"]:
            print(f"Solution: {value['solution_path']}")
        
        if "messages" in value and value["messages"]:
            msg = value["messages"][-1]
            content = msg.content if hasattr(msg, 'content') else str(msg)
            print(f"Output: {content[:300]}...")
        
        if "next" in value:
            print(f"Next: {value['next']}")
        
        if "best_eval_score" in value and value["best_eval_score"] is not None:
            _final_best_score = value["best_eval_score"]
        if "best_submission_path" in value and value["best_submission_path"]:
            _final_best_sub = value["best_submission_path"]
        if "submission_uploaded" in value and value["submission_uploaded"]:
            _final_uploaded = True

print(f"\n{'='*60}")
print("Workflow Complete!")
print(f"{'='*60}")

print(f"\n--- SUBMISSION LOG ---")
print(f"Best-ever MSE: {_final_best_score}")
print(f"Threshold: {MSE_THRESHOLD}")
print(f"Passed: {_final_best_score is not None and _final_best_score <= MSE_THRESHOLD}")
print(f"Uploaded to Kaggle: {_final_uploaded}")
if _final_best_sub:
    print(f"Best submission file: {_final_best_sub}")
    try:
        from pathlib import Path
        import pandas as _pd
        _p = Path(_final_best_sub)
        if _p.exists():
            _df = _pd.read_csv(_p)
            print(f"  Shape: {_df.shape}")
            print(f"  Columns: {list(_df.columns)}")
            pred_col = 'prediction' if 'prediction' in _df.columns else _df.columns[-1]
            print(f"  Prediction range: {_df[pred_col].min():.2f} to {_df[pred_col].max():.2f}")
    except Exception as e:
        print(f"  (could not read: {e})")
if eval_history:
        history_str = " -> ".join(str(round(e["mse"])) for e in eval_history)
        print(f"Score history: {history_str}")
print("--- END LOG ---")

print()
print(monitor.get_summary())

Starting Workflow
Competition: mws-ai-agents-2026
  [MONITOR] ℹ️ [supervisor] start: iteration=0

👁️  Supervisor (iteration 0)
   Dataset: EMPTY
   EDA: not done, Solution: 
   Validated: False, Submission: 
   Eval: score=None, passed=None, best_ever=None
   Uploaded: False
   ➡️  Next: kaggle_agent
   📝 Download the competition dataset files for 'mws-ai-agents-2026' from Kaggle
  [MONITOR] ℹ️ [supervisor] complete: duration=0.6s next=kaggle_agent

Node: supervisor
Output: Download the competition dataset files for 'mws-ai-agents-2026' from Kaggle...
Next: kaggle_agent
  [MONITOR] ℹ️ [kaggle_agent] start: iteration=1
🔧 Executing tool: download_competition_files
📥 Downloading 'mws-ai-agents-2026'...
✅ Downloaded 4 files to C:\Users\Сергей\Downloads\competitions\mws-ai-agents-2026
🗂️  Paths in state: ['sample_submition', 'solution', 'test', 'train']
  [MONITOR] ℹ️ [kaggle_agent] complete: duration=3.0s next=supervisor

Node: kaggle_agent
Dataset: ['sample_submition', 'solution', 'test',

C:\Users\Сергей\AppData\Local\Temp\ipykernel_43632\625027310.py:131: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for bin_name, group in merged.groupby('target_bin'):


  📊 Eval strategy RAG query: error bin imbalance worst bin (232.75, 365.0] targeted feature engineering
  📊 Retrieved 4 eval strategies
  [MONITOR] ℹ️ [evaluator] complete: duration=3.9s next=supervisor

Node: evaluator_agent
  [LLM START] Calling qwen/qwen3-32b
  [LLM DECISION] Selected tools: ['validate_submission_format']
    -> validate_submission_format({"submission_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\submission_solution_mws_ai_agents_2026.csv"})
  [LLM START] Calling qwen/qwen3-32b
  [LLM DECISION] Selected tools: ['score_submission']
    -> score_submission({"solution_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\solution.csv", "submission_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\submission_solutio)
  [TOOL CALL] validate_submission_format({"submission_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\submission_solution_mws_ai_agents_2026.csv"})
  [TOOL RESULT] val

C:\Users\Сергей\AppData\Local\Temp\ipykernel_43632\625027310.py:131: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for bin_name, group in merged.groupby('target_bin'):


  📊 Eval strategy RAG query: MSE regressed by 3.6% clipping range worst_under 501 underprediction error bin imbalance worst bin (232.75, 365.0] targe
  📊 Retrieved 4 eval strategies
  [MONITOR] ℹ️ [evaluator] complete: duration=3.5s next=supervisor

Node: evaluator_agent
  [LLM START] Calling qwen/qwen3-32b
  [LLM DECISION] Selected tools: ['validate_submission_format']
    -> validate_submission_format({"submission_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\submission_solution_mws_ai_agents_2026.csv"})
  [LLM START] Calling qwen/qwen3-32b
  [LLM DECISION] Selected tools: ['score_submission']
    -> score_submission({"solution_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\solution.csv", "submission_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\submission_solutio)
  [TOOL CALL] validate_submission_format({"submission_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\submission_solution

C:\Users\Сергей\AppData\Local\Temp\ipykernel_43632\625027310.py:131: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for bin_name, group in merged.groupby('target_bin'):


  📊 Eval strategy RAG query: MSE regressed by 3.4% clipping range worst_under 514 underprediction error bin imbalance worst bin (232.75, 365.0] targe
  📊 Retrieved 4 eval strategies
  [MONITOR] ℹ️ [evaluator] complete: duration=3.7s next=supervisor

Node: evaluator_agent
  [LLM START] Calling qwen/qwen3-32b
  [LLM DECISION] Selected tools: ['validate_submission_format']
    -> validate_submission_format({"submission_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\submission_solution_mws_ai_agents_2026.csv"})
  [LLM START] Calling qwen/qwen3-32b
  [LLM DECISION] Selected tools: ['score_submission']
    -> score_submission({"solution_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\solution.csv", "submission_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\submission_solutio)
  [TOOL CALL] validate_submission_format({"submission_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\submission_solution

C:\Users\Сергей\AppData\Local\Temp\ipykernel_43632\625027310.py:131: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for bin_name, group in merged.groupby('target_bin'):


  📊 Eval strategy RAG query: MSE improved by 8.0% error bin imbalance worst bin (232.75, 365.0] targeted feature engineering
  📊 Retrieved 4 eval strategies
  [MONITOR] ℹ️ [evaluator] complete: duration=3.5s next=supervisor

Node: evaluator_agent
  [LLM START] Calling qwen/qwen3-32b
  [LLM DECISION] Selected tools: ['validate_submission_format']
    -> validate_submission_format({"submission_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\submission_solution_candidate_2.csv"})
  [LLM START] Calling qwen/qwen3-32b
  [LLM DECISION] Selected tools: ['score_submission']
    -> score_submission({"solution_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\solution.csv", "submission_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\submission_solutio)
  [TOOL CALL] validate_submission_format({"submission_path": "C:\\Users\\Сергей\\Downloads\\competitions\\mws-ai-agents-2026\\submission_solution_candidate_2.csv"})
  [TOOL RESU